In [1]:
from google.colab import userdata
import os

aws_dir = os.path.expanduser("~/.aws")
os.makedirs(aws_dir, exist_ok=True)

key_id = userdata.get('AWS_ACCESS_KEY_ID')
secret_key = userdata.get('AWS_SECRET_ACCESS_KEY')

with open(f"{aws_dir}/credentials", "w") as f:
    f.write(f"[default]\naws_access_key_id = {key_id}\naws_secret_access_key = {secret_key}\n")

with open(f"{aws_dir}/config", "w") as f:
    f.write("[default]\nregion = us-east-1\n")

print("Successfully configured ~/.aws/credentials and ~/.aws/config")

Successfully configured ~/.aws/credentials and ~/.aws/config


In [2]:
!apt update -q && apt-get install -qy openjdk-21-jdk-headless > /dev/null 2>&1

import subprocess
from IPython.core.magic import register_cell_magic

class PersistentJShell:
    def __init__(self):
        self.proc = subprocess.Popen(
            ["jshell", "-s"],
            stdin=subprocess.PIPE,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1
        )

    def execute(self, code):
        if self.proc.poll() is not None:
            raise RuntimeError("JShell process has terminated.")

        marker = "___JSHELL_CELL_END___"
        payload = code.strip() + f'\nSystem.out.println("{marker}");\n'
        self.proc.stdin.write(payload)
        self.proc.stdin.flush()

        lines = []
        while True:
            line = self.proc.stdout.readline()
            if not line:
                break
            if marker in line:
                break
            lines.append(line)

        return "".join(lines)

_jshell_instance = PersistentJShell()

@register_cell_magic
def jshell(line, cell):
    out = _jshell_instance.execute(cell)
    if out:
        print(out, end="")


Hit:1 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:4 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Hit:6 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Fetched 3,917 B in 1s (5,158 B/s)
Reading package lists...
Building dependency tree...
Reading state information...
55 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [3]:
%%jshell

void runPy(String code) throws Exception {
    String marker = "___END_OF_PYTHON_CELL___";

    // Base64 encode code payload to bypass all quote & newline syntax errors
    String b64 = java.util.Base64.getEncoder().encodeToString(code.getBytes(java.nio.charset.StandardCharsets.UTF_8));

    // Guarantee marker printing using try...finally and display trailing expressions using ast
    String payload = String.format(
        "import base64, ast, traceback\n" +
        "try:\n" +
        "    code_str = base64.b64decode('%s').decode('utf-8')\n" +
        "    tree = ast.parse(code_str)\n" +
        "    if tree.body and isinstance(tree.body[-1], ast.Expr):\n" +
        "        exec(compile(ast.Module(body=tree.body[:-1], type_ignores=[]), '<string>', 'exec'), globals())\n" +
        "        res = eval(compile(ast.Expression(body=tree.body[-1].value), '<string>', 'eval'), globals())\n" +
        "        if res is not None:\n" +
        "            print(res)\n" +
        "    else:\n" +
        "        exec(compile(tree, '<string>', 'exec'), globals())\n" +
        "except Exception:\n" +
        "    traceback.print_exc()\n" +
        "finally:\n" +
        "    print('%s', flush=True)\n",
        b64, marker
    );

    pyIn.println(payload);

    String line;
    while ((line = pyOut.readLine()) != null) {
        if (line.contains(marker)) break;
        System.out.println(line);
    }
}

import java.io.BufferedReader;
import java.io.InputStreamReader;
import java.io.PrintStream;
import java.io.IOException;

Process pythonProcess;
PrintStream pyIn;
BufferedReader pyOut;

try {
    ProcessBuilder pb = new ProcessBuilder("python3", "-i", "-u");
    // Redirect stderr to stdout to prevent pipe deadlock and capture Python errors
    pb.redirectErrorStream(true);
    pythonProcess = pb.start();

    pyIn = new PrintStream(pythonProcess.getOutputStream(), true);
    pyOut = new BufferedReader(new InputStreamReader(pythonProcess.getInputStream()));

    System.out.println("Python interpreter initialized safely.");
} catch (IOException e) {
    e.printStackTrace();
}

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
->  void runPy(String code) throws Exception {
>>         String marker = "___END_OF_PYTHON_CELL___";
>>     
>>         // Base64 encode code payload to bypass all quote & newline syntax errors
>>          String b64 = java.util.Base64.getEncoder().encodeToString(code.getBytes(java.nio.charset.StandardCharsets.UTF_8));
>>      
>>         // Guarantee marker printing using try...finally and display trailing expressions using ast
>>         String payload = String.format(
>>              "import base64, ast, traceback\n" +
>>             "try:\n" +
>>             "    code_str = base64.b64decode('%s').decode('utf-8')\n" +
>>              "    tree = ast.parse(cod

In [4]:
%%jshell

runPy("""
# Google Colab setup: fetch this repository and use this notebook's directory.
from pathlib import Path ; from IPython.display import display, HTML
import os
os.system('pip install boto3');

try:
    from google.colab import userdata
except ImportError:
    userdata = None

def get_colab_secret(name, required=True):
    try:
        value = os.environ.get(name) or userdata.get(name)
    except Exception as exc:
        if required:
            raise RuntimeError(f'Unable to read Colab Secret: {name}') from exc
        return None
    if required and not value:
        raise RuntimeError(f'Add the Colab Secret {name} and grant this notebook access.')
    return value

REPO_ROOT = Path('/content/BITS_programming')
if not REPO_ROOT.exists():
    os.system('git clone https://github.com/aqwertyuiop48/BITS_programming.git /content/BITS_programming');

NOTEBOOK_DIR = REPO_ROOT / 'assignments/assignment_20'
os.chdir(NOTEBOOK_DIR)
print(f'Working directory: {NOTEBOOK_DIR}')
""");

->  Python 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0] on linux
Type "help", "copyright", "credits" or "license" for more information.
>>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Requirement already satisfied: boto3 in /usr/local/lib/python3.13/dist-packages (1.43.105)
Working directory: /content/BITS_programming/assignments/assignment_20


# Lesson 5 — Amazon EKS End-to-End Lab

Real Kubernetes on real EKS — from cluster bootstrap to blue/green cutover, with an honest teardown.

## Read this first

This notebook drives Amazon EKS through `boto3` + shell invocations of `aws`, `eksctl`, and `kubectl`. Every managed resource is a **real AWS API call** — no mocks, no local Minikube fallback. That means:

- **`eksctl create cluster` provisions a real CloudFormation stack** (VPC, IAM roles, control plane, node group). ~15 min.
- **The control plane bills at $0.10/hr** the moment it reaches `ACTIVE`.
- **One `t3.medium` worker bills at ~$0.042/hr.** LoadBalancer Service adds ~$0.025/hr for an NLB.
- **Blue/green + load test can push node count to 3** briefly — bump the top of the cost estimate to $0.20/hr while the lab is running.

## Verification status — what was actually checked

Structural checks run against the reference SDK versions (`boto3 1.43+`, `sagemaker` not required here):

- All 40+ code cells parse — 0 syntax errors
- All boto3 client factories succeed
- Every AWS operation is idempotent (check-then-create or `--ignore-not-found` on delete)
- Every YAML manifest generated by cells passes `yaml.safe_load`
- Preflight (Block 0) verifies quotas + tools + region before you sink 15 min into cluster provisioning

**Not verified end-to-end:** the actual `eksctl create cluster` and its downstream steps require live AWS credentials + quota. Expect the usual first-pass IAM debugging (EKS needs 4 IAM policies + one for CloudFormation). Block 0 catches the common causes up front.

## What you'll build

| Stage | AWS service | K8s objects | Duration |
|-------|-------------|-------------|----------|
| 0     | Preflight — read-only checks           | —                              | 30s |
| 1     | AWS session + config                    | —                              | 30s |
| 2     | `eksctl create cluster`                 | Cluster, node group             | ~15 min |
| 3     | Deploy v1                               | Namespace, ConfigMap, Secret, Deployment, Service | ~2 min |
| 4     | Autoscaling + load test                 | HPA, metrics-server, load pod   | ~5 min |
| 5     | Rolling update v1 → v2                  | Deployment (updated)            | ~2 min |
| 6     | Blue/green cutover to v3                | Second Deployment + Service switch | ~3 min |
| 7     | External LoadBalancer                   | Service (type=LoadBalancer)     | ~3 min |
| 8     | Cleanup (guarded — CLEANUP=True)        | All + `eksctl delete cluster`   | ~10 min |

**Total cost if you run once and clean up promptly: ~$0.20 – $0.35.**

## 0. Preflight — run this FIRST

Fails fast on the things that would otherwise waste 15+ minutes waiting for a cluster to provision before you discover the problem. Nothing here mutates AWS.

In [5]:
%%jshell

runPy("""
# Block 0b - Install kubectl and eksctl if missing. Skip if Block 0 passed.
import shutil, subprocess, os

if shutil.which("kubectl") is None:
    print("Installing kubectl...")
    subprocess.run(
        '''
        curl -sSLO "https://dl.k8s.io/release/$(curl -sL https://dl.k8s.io/release/stable.txt)/bin/linux/amd64/kubectl"
        chmod +x kubectl
        mkdir -p ~/.local/bin
        mv kubectl ~/.local/bin/
        ''',
        shell=True, check=False,
    )
    print("kubectl installed to ~/.local/bin/")

if shutil.which("eksctl") is None:
    print("Installing eksctl...")
    subprocess.run(
        '''
        PLATFORM=$(uname -s)_amd64
        curl -sSL "https://github.com/eksctl-io/eksctl/releases/latest/download/eksctl_$PLATFORM.tar.gz" \
            | tar xz -C /tmp
        mkdir -p ~/.local/bin
        mv /tmp/eksctl ~/.local/bin/
        ''',
        shell=True, check=False,
    )
    print("eksctl installed to ~/.local/bin/")

# Make sure ~/.local/bin is on PATH for THIS Python session
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ.get("PATH", "")
print("PATH updated. Re-run Block 0 to verify.")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Installing kubectl...
kubectl installed to ~/.local/bin/
Installing eksctl...
eksctl installed to ~/.local/bin/
PATH updated. Re-run Block 0 to verify.


### Fix for Preflight Errors: Install AWS CLI and Configure Credentials

The `Block 0` preflight check failed because:
1.  The `aws` command-line interface was not found.
2.  AWS credentials (Access Key ID, Secret Access Key, Session Token) were not set as environment variables.

The following cells will address these issues. After running them, re-run `Block 0` (cell `10a33c79`).

In [6]:
%%jshell

runPy("""
# Install awscli
print("Installing awscli...")
os.system('pip install awscli');

# Ensure ~/.local/bin is on PATH for THIS Python session if not already
import os
if os.path.expanduser("~/.local/bin") not in os.environ["PATH"]:
    os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ.get("PATH", "")
    print("PATH updated to include ~/.local/bin")
else:
    print("~/.local/bin already in PATH")

print("awscli installed.")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Installing awscli...
~/.local/bin already in PATH
awscli installed.


In [7]:
%%jshell

runPy("""
# Block 2 - Create AWS clients

os.system("pip install boto3")
import boto3

session = boto3.Session()
region  = session.region_name or "us-east-2"

s3           = session.client("s3",         region_name=region)
ecr          = session.client("ecr",        region_name=region)
codebuild    = session.client("codebuild",  region_name=region)
ecs          = session.client("ecs",        region_name=region)
logs_client  = session.client("logs",       region_name=region)
cloudwatch   = session.client("cloudwatch", region_name=region)
sts          = session.client("sts",        region_name=region)
iam          = session.client("iam",        region_name=region)

identity   = sts.get_caller_identity()
account_id = identity["Account"]
caller_arn = identity["Arn"]

# Determine the execution_role_arn
# If it's the root user, directly use caller_arn (not recommended for service roles)
# If it's an assumed role or direct IAM role, use iam.get_role to get the exact ARN.

if caller_arn.endswith(':root'):
    execution_role_arn = caller_arn
    print("Warning: Running as root user. Services requiring a dedicated IAM role (e.g., CodeBuild, ECS) may encounter issues or require specific configurations that are not recommended for root access.")
elif ":assumed-role/" in caller_arn:
    role_name = caller_arn.split(":assumed-role/")[1].split("/")[0]
    # Use iam.get_role() to get the EXACT ARN including /service-role/ path
    execution_role_arn = iam.get_role(RoleName=role_name)["Role"]["Arn"]
elif ":role/" in caller_arn: # Direct IAM role ARN
    role_name = caller_arn.split(":role/")[-1]
    execution_role_arn = iam.get_role(RoleName=role_name)["Role"]["Arn"]
else: # Likely an IAM user
    execution_role_arn = caller_arn
    print("Warning: Running as an IAM user. Services requiring a dedicated IAM role (e.g., CodeBuild, ECS) may encounter issues or require specific configurations. Consider configuring an IAM role.")

print("Region         :", region)
print("Account        :", account_id)
print("Caller ARN     :", caller_arn)
print("Execution role :", execution_role_arn)
""");

->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Requirement already satisfied: boto3 in /usr/local/lib/python3.13/dist-packages (1.43.105)
Region         : us-east-1
Account        : 455865672536
Caller ARN     : arn:aws:iam::455865672536:root
Execution role : arn:aws:iam::455865672536:root


In [8]:
%%jshell

runPy("""
# Block 0 - PREFLIGHT. Run before anything else. Read-only.
import sys, subprocess, shutil, json, os, pathlib
from importlib.metadata import version, PackageNotFoundError

# --- Ensure PATH for locally installed CLI tools ------------------------------
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ.get("PATH", "")


problems  = []
warnings_ = []

# --- 1. Python packages -------------------------------------------------------
for pkg in ["boto3", "pyyaml"]:
    try:
        print(f"PASS  {pkg:10s} {version(pkg)}")
    except PackageNotFoundError:
        problems.append(f"{pkg} not installed. FIX: %pip install -q {pkg}")

# --- 2. External CLI tools ----------------------------------------------------
# EKS work is CLI-driven. If any of these are missing, Blocks 6+ fail hard.
def _cli_ok(name, version_flag="--version"):
    if shutil.which(name) is None:
        return None
    try:
        r = subprocess.run([name] + version_flag.split(), capture_output=True, text=True, timeout=10)
        return (r.stdout or r.stderr).strip().split("\\n")[0][:80]
    except Exception as e:
        return f"error: {e}"

for tool, flag in [("aws", "--version"), ("kubectl", "version --client=true -o=json"),
                   ("eksctl", "version"), ("curl", "--version")]:
    v = _cli_ok(tool, flag)
    if v is None:
        problems.append(f"{tool} not found on PATH. See Block 0b for install commands.")
    else:
        print(f"PASS  {tool:10s} {v[:60]}")

# --- 3. AWS credentials + identity --------------------------------------------
# NoCredentialsError is NOT a ClientError subclass, so we catch broadly.
try:
    import boto3
    from botocore.exceptions import ClientError, NoCredentialsError
    _sts = boto3.client("sts")
    _id  = _sts.get_caller_identity()
    print(f"PASS  identity   {_id['Arn']}")
    _account = _id["Account"]
except NoCredentialsError:
    problems.append(
        "AWS credentials not found. FIX: run `aws configure` or export "
        "AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY / AWS_SESSION_TOKEN."
    )
    _account = None
except Exception as e:
    problems.append(f"Cannot resolve AWS identity: {type(e).__name__}: {e}")
    _account = None

# --- 4. Region -----------------------------------------------------------------
# EKS is available in every commercial region, but eksctl defaults to us-west-2.
# Being explicit avoids "cluster created in wrong region" surprises.
_region = boto3.Session().region_name or "us-east-2"
print(f"PASS  region     {_region}")

# --- 5. EC2 quota for the worker node type ------------------------------------
# t3.medium is our worker. The EC2 quota code for on-demand standard instances:
# L-1216C47A = "Running On-Demand Standard (A, C, D, H, I, M, R, T, Z) instances"
# Sandbox accounts sometimes cap this at 0 or 5.
try:
    _sq = boto3.client("service-quotas", region_name=_region)
    q = _sq.get_service_quota(ServiceCode="ec2", QuotaCode="L-1216C47A")
    quota_val = q["Quota"]["Value"]
    if quota_val < 4:
        warnings_.append(
            f"EC2 standard on-demand quota is only {int(quota_val)} vCPUs. "
            f"1x t3.medium = 2 vCPUs, so this is tight. If Section 4 tries to scale up, it may fail."
        )
    else:
        print(f"PASS  ec2 quota  {int(quota_val)} vCPUs available for standard on-demand")
except Exception as e:
    warnings_.append(f"Could not check EC2 quota: {e}. Assuming >= 4 vCPUs and continuing.")

# --- 6. IAM permission smoke-test ---------------------------------------------
# eksctl needs EKS + EC2 + CloudFormation + IAM. Full simulate is overkill;
# we do a cheap describe call on each service.
if _account is not None:
    try:
        boto3.client("eks",            region_name=_region).list_clusters(maxResults=1)
        boto3.client("cloudformation", region_name=_region).list_stacks(StackStatusFilter=["CREATE_COMPLETE"])
        boto3.client("ec2",            region_name=_region).describe_vpcs(MaxResults=5)
        boto3.client("elbv2",          region_name=_region).describe_load_balancers(PageSize=1)
        print("PASS  permissions eks + cloudformation + ec2 + elbv2 describe calls all succeed")
    except ClientError as e:
        err_code = e.response.get("Error", {}).get("Code", "?")
        if err_code in ("AccessDenied", "UnauthorizedOperation"):
            problems.append(
                f"IAM permission missing: {err_code}. Attach AmazonEKSClusterPolicy, "
                f"AmazonEKSServicePolicy, AmazonEC2FullAccess, and CloudFormationFullAccess "
                f"to your caller identity."
            )
        else:
            warnings_.append(f"Unexpected error during permission check: {err_code}")
    except Exception as e:
        warnings_.append(f"Permission check inconclusive: {type(e).__name__}: {e}")

# --- 7. Existing cluster with the same name -----------------------------------
# Block 6 uses a deterministic cluster name based on account/region. If a stale
# one is running (from a previous lab), we say so before we try to create another.
try:
    _existing = boto3.client("eks", region_name=_region).list_clusters()["clusters"]
    stale = [c for c in _existing if c.startswith("eks-lab-")]
    if stale:
        warnings_.append(f"Existing lab cluster(s) found: {stale}. Section 8 (cleanup) can remove them.")
except Exception:
    pass

print()
if problems:
    print("BLOCK — do not continue. Fix these:")
    for p in problems: print(f"  - {p}")
    raise SystemExit("Preflight failed. See messages above.")
if warnings_:
    print("Warnings (non-blocking):")
    for w in warnings_: print(f"  ! {w}")
    print()

print("Preflight complete. Estimated total cost if you clean up promptly: $0.20 - $0.35.")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... PASS  boto3      1.43.105
PASS  pyyaml     6.0.3
PASS  aws        aws-cli/1.46.1 Python/3.13.15 Linux/6.6.122+ botocore/1.43.6
PASS  kubectl    {
PASS  eksctl     0.230.0
PASS  curl       curl 8.5.0 (x86_64-pc-linux-gnu) libcurl/8.5.0 OpenSSL/3.0.1
PASS  identity   arn:aws:iam::455865672536:root
PASS  region     us-east-1
PASS  ec2 quota  16 vCPUs available for standard on-demand
PASS  permissions eks + cloudformation + ec2 + elbv2 describe calls all succeed

Preflight complete. Estimated total cost if you clean up promptly: $0.20 - $0.35.


### Block 0b — Install missing CLI tools

Only run the cell below if Block 0 flagged a missing tool. Skip otherwise.

## 1. Environment Setup

Imports, clients, and the deterministic naming scheme that makes every operation in the notebook idempotent.

In [9]:
%%jshell

runPy("""
# Block 1 - Imports
import io, os, json, time, subprocess, shutil, base64, hashlib
import datetime as dt
from pathlib import Path ; from IPython.display import display, HTML

import boto3
import yaml
from botocore.exceptions import ClientError

print("Libraries imported")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Libraries imported


In [10]:
%%jshell

runPy("""
# Block 2 - Load credentials and region directly from ~/.aws/config and ~/.aws/credentials
import os, pathlib, configparser, boto3

# 1. Verify that the AWS credentials file exists and is not empty
aws_dir = pathlib.Path.home() / ".aws"
credentials_file = aws_dir / "credentials"
config_file = aws_dir / "config"

if not credentials_file.exists() or credentials_file.stat().st_size == 0:
    raise FileNotFoundError("~/.aws/credentials file is missing or empty. Please write your [default] profile first.")

# 2. Clear explicit auth environment variables so boto3/CLI are forced to use file profile credentials
for env_var in ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_SESSION_TOKEN"]:
    os.environ.pop(env_var, None)

# 3. Parse region from ~/.aws/config if present, defaulting to 'us-east-2'
profile_config = configparser.ConfigParser()
profile_config.read(config_file)
region = profile_config.get("default", "region", fallback=os.environ.get("AWS_DEFAULT_REGION", "us-east-2"))

os.environ["AWS_DEFAULT_REGION"] = region
os.environ["AWS_REGION"] = region

# 4. Initialize boto3 session using the default profile from ~/.aws/credentials
boto_session = boto3.Session(profile_name="default", region_name=region)

clients = {svc: boto_session.client(svc, region_name=region) for svc in ["sts", "eks", "ec2", "elbv2", "cloudformation", "iam"]}
sts, eks, ec2, elbv2, cfn, iam = (clients[s] for s in ["sts", "eks", "ec2", "elbv2", "cloudformation", "iam"])

# 5. Fetch caller identity using file credentials
identity = sts.get_caller_identity()
account_id = identity["Account"]

print(f"Region: {region} | Account: {account_id} | Caller: {identity['Arn']}")
""");

->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Region: us-east-1 | Account: 455865672536 | Caller: arn:aws:iam::455865672536:root


In [11]:
%%jshell

runPy("""
# Block 3 - Project configuration
AWS_REGION_NAME='us-east-2'; os.environ['AWS_DEFAULT_REGION']=AWS_REGION_NAME; os.environ['AWS_REGION']=AWS_REGION_NAME
PROJECT_NAME='eks-lab'; ENVIRONMENT='demo'; ALLOW_MUTATIONS=True; CLEANUP=False
NODE_INSTANCE_TYPE='t3.medium'; NODE_MIN,NODE_DESIRED,NODE_MAX=1,1,3; K8S_VERSION='1.31'
cluster_name=f'{PROJECT_NAME}-{ENVIRONMENT}'; nodegroup_name='ng-primary'; namespace='ml-lab'; service_account='inference-sa'
APP_IMAGE='hashicorp/http-echo:1.0'; APP_PORT=5678; APP_TEXT_V1='inference-server-v1'; APP_TEXT_V2='inference-server-v2'; APP_TEXT_V3_BG='inference-server-v3-green'
work_dir=Path('./eks_work'); work_dir.mkdir(exist_ok=True)
boto_session=boto3.Session(region_name=AWS_REGION_NAME); region=AWS_REGION_NAME
sts=boto_session.client('sts',region_name=region); eks=boto_session.client('eks',region_name=region); ec2=boto_session.client('ec2',region_name=region); elbv2=boto_session.client('elbv2',region_name=region); cfn=boto_session.client('cloudformation',region_name=region); iam=boto_session.client('iam',region_name=region)
identity=sts.get_caller_identity(); account_id=identity['Account']
print(f'Cluster: {cluster_name} | Region: {region} | Account: {account_id}')
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Cluster: eks-lab-demo | Region: us-east-2 | Account: 455865672536


## Optional: Assume an IAM Role

If you're using the AWS root user (as indicated by `arn:aws:iam::061831608851:root` in `Block 2`), it's highly recommended to switch to an IAM role with specific permissions. This improves security by enforcing the principle of least privilege.

To assume a role, you need to:
1.  **Create an IAM Role** in your AWS account with the necessary permissions (e.g., EKS, EC2, CloudFormation, IAM access for this lab). This role's trust policy must allow your current principal (e.g., your Colab instance role or your root user) to assume it.
2.  **Add the Role ARN** to Colab secrets as `AWS_ROLE_ARN`.
3.  **Run the cell below** to assume the role and update your session credentials.

After running this cell, **re-run `Block 2` and `Block 3b`** to ensure the new assumed role's identity and permissions are reflected and verified.

In [12]:
%%jshell

runPy("""
# Block 3c - Assume IAM Role (Reads role ARN from ~/.aws/config or environment)
import os, pathlib, configparser, boto3
from botocore.exceptions import ClientError

# 1. Resolve AWS_ROLE_ARN from environment variable or ~/.aws/config
aws_dir = pathlib.Path.home() / ".aws"
config_file = aws_dir / "config"

role_config = configparser.ConfigParser()
if config_file.exists():
    role_config.read(config_file)

role_arn = os.environ.get("AWS_ROLE_ARN") or role_config.get("default", "role_arn", fallback=None)

# 2. Assume Role if present, update environment and write temporary credentials to ~/.aws/credentials
if role_arn:
    print(f"Attempting to assume role: {role_arn}")
    try:
        # Use existing STS client or fallback to new session STS
        sts_client = globals().get('sts') or boto3.Session().client('sts', region_name=os.environ.get('AWS_DEFAULT_REGION', 'us-east-2'))
        response = sts_client.assume_role(RoleArn=role_arn, RoleSessionName='ColabEksLabSession')
        creds = response['Credentials']

        # Update environment variables for subprocesses / CLI tools
        os.environ['AWS_ACCESS_KEY_ID'] = creds['AccessKeyId']
        os.environ['AWS_SECRET_ACCESS_KEY'] = creds['SecretAccessKey']
        os.environ['AWS_SESSION_TOKEN'] = creds['SessionToken']

        # Update ~/.aws/credentials with assumed temporary credentials
        credentials_file = aws_dir / "credentials"
        cred_parser = configparser.ConfigParser()
        if credentials_file.exists():
            cred_parser.read(credentials_file)

        if 'default' not in cred_parser.sections():
            cred_parser.add_section('default')

        cred_parser.set('default', 'aws_access_key_id', creds['AccessKeyId'])
        cred_parser.set('default', 'aws_secret_access_key', creds['SecretAccessKey'])
        cred_parser.set('default', 'aws_session_token', creds['SessionToken'])

        with open(credentials_file, 'w') as f:
            cred_parser.write(f)

        # Re-initialize boto3 session and client dictionary mapping
        region = os.environ.get('AWS_DEFAULT_REGION', 'us-east-2')
        boto_session = boto3.Session(region_name=region)

        clients = {svc: boto_session.client(svc, region_name=region) for svc in ['sts', 'eks', 'ec2', 'elbv2', 'cloudformation', 'iam']}
        sts, eks, ec2, elbv2, cfn, iam = (clients[s] for s in ['sts', 'eks', 'ec2', 'elbv2', 'cloudformation', 'iam'])

        # Inject updated clients into python global scope
        globals().update(clients)

        identity = sts.get_caller_identity()
        print(f"Successfully assumed role. New identity: {identity['Arn']}")
    except ClientError as e:
        print(f"Error assuming role {role_arn}: {e}")
        print("Please check the Role ARN and the trust policy of the role.")
    except Exception as e:
        print(f"An unexpected error occurred during role assumption: {e}")
else:
    print("AWS_ROLE_ARN not found in environment or ~/.aws/config. Skipping role assumption.")
    print("To assume a role, set role_arn in ~/.aws/config under [default] or export AWS_ROLE_ARN.")
""");

->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... AWS_ROLE_ARN not found in environment or ~/.aws/config. Skipping role assumption.
To assume a role, set role_arn in ~/.aws/config under [default] or export AWS_ROLE_ARN.


### How to Revert to Original Identity

If you have assumed an IAM role and wish to switch back to your original credentials (e.g., the root user or initial IAM user you started with), you need to re-load those credentials and re-initialize the AWS clients.

In [13]:
%%jshell

runPy("""
# Block 3d - Revert to Original AWS Identity from ~/.aws/credentials
# Restores the default profile credentials from file and clears assumed session tokens.

import os, pathlib, configparser, boto3

print("Reverting to original AWS identity...")

aws_dir = pathlib.Path.home() / ".aws"
credentials_file = aws_dir / "credentials"
config_file = aws_dir / "config"

# 1. Clear temporary session credentials from environment variables
for env_var in ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_SESSION_TOKEN"]:
    os.environ.pop(env_var, None)

# 2. Clean up session token entry from ~/.aws/credentials if it exists
if credentials_file.exists():
    cred_parser = configparser.ConfigParser()
    cred_parser.read(credentials_file)
    if 'default' in cred_parser.sections() and cred_parser.has_option('default', 'aws_session_token'):
        cred_parser.remove_option('default', 'aws_session_token')
        with open(credentials_file, 'w') as f:
            cred_parser.write(f)

# 3. Resolve active region
region = os.environ.get('AWS_DEFAULT_REGION', 'us-east-2')

# 4. Re-initialize boto3 session and clients reading from default profile
boto_session = boto3.Session(profile_name="default", region_name=region)

clients = {svc: boto_session.client(svc, region_name=region) for svc in ['sts', 'eks', 'ec2', 'elbv2', 'cloudformation', 'iam']}
sts, eks, ec2, elbv2, cfn, iam = (clients[s] for s in ['sts', 'eks', 'ec2', 'elbv2', 'cloudformation', 'iam'])

# Inject updated clients into global scope
globals().update(clients)

_current_identity = sts.get_caller_identity()
print(f"Successfully reverted to original identity: {_current_identity['Arn']}")
""");

->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Reverting to original AWS identity...
Successfully reverted to original identity: arn:aws:iam::455865672536:root


In [14]:
%%jshell

runPy("""
# Block 3b - REAL preflight check (no simulation)
# Calls real AWS APIs to verify permissions. Read-only calls only:
# no resources created, no cost, no side effects. If a call succeeds,
# the permission works. If it fails, you get the actual AWS error
# code - the same one Block 5 would return.
#
# Why not simulate_principal_policy? The IAM policy simulator produces
# false negatives for some EKS actions (list/create) even when the
# actual API would succeed. Real calls are authoritative.
from botocore.exceptions import ClientError

def _role_name_from_arn(arn):
    return arn.split("/")[1] if ":assumed-role/" in arn else arn.rsplit("/", 1)[-1]

_caller = sts.get_caller_identity()
_role_name = _role_name_from_arn(_caller["Arn"])
print(f"Testing real AWS calls as: {_caller['Arn']}\\n")

_checks = [
    ("eks:ListClusters",          lambda: eks.list_clusters()),
    ("ec2:DescribeVpcs",          lambda: ec2.describe_vpcs(MaxResults=5)),
    ("ec2:DescribeSubnets",       lambda: ec2.describe_subnets(MaxResults=5)),
    ("cloudformation:ListStacks", lambda: cfn.list_stacks()),
]

# Conditionally add the iam:GetRole check, skipping if the caller is the root user
if ":root" not in _caller["Arn"]:
    _checks.append(("iam:GetRole (self)", lambda: iam.get_role(RoleName=_role_name)))
else:
    print(f"  SKIP  iam:GetRole (self) (Not applicable for root user identity)")

_failed = []
for _name, _fn in _checks:
    try:
        _fn()
        print(f"  OK    {_name}")
    except ClientError as _e:
        _code = _e.response["Error"]["Code"]
        print(f"  FAIL  {_name}  ({_code})")
        _failed.append((_name, _code))
    except Exception as _e:
        print(f"  FAIL  {_name}  ({type(_e).__name__})")
        _failed.append((_name, type(_e).__name__))

if _failed:
    print(f"\\n{len(_failed)} check(s) failed. Attach missing policies to role '{_role_name}':")
    print("  1. AmazonEKSClusterPolicy")
    print("  2. AmazonEKSWorkerNodePolicy")
    print("  3. AmazonEKSServicePolicy")
    print("  4. AmazonEC2FullAccess")
    print("  5. AWSCloudFormationFullAccess")
    print("  6. IAMFullAccess")
    print("Wait 60 seconds after attaching, then re-run this cell.")
    raise RuntimeError(f"Failed checks: {[n for n,_ in _failed]}")
else:
    print("\\nAll real API calls succeeded. Safe to proceed.")
    print("Note: eks:CreateCluster is not tested here (that would actually create a cluster).")
    print("      If eks:ListClusters works, CreateCluster is in the same policy and should too.")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Testing real AWS calls as: arn:aws:iam::455865672536:root

  SKIP  iam:GetRole (self) (Not applicable for root user identity)
  OK    eks:ListClusters
  OK    ec2:DescribeVpcs
  OK    ec2:DescribeSubnets
  OK    cloudformation:ListStacks

All real API calls succeeded. Safe to proceed.
Note: eks:CreateCluster is not tested here (that would actually create a cluster).
      If eks:ListClusters works, CreateCluster is in the same policy and should too.


In [15]:
%%jshell

runPy("""
# Block 4 - Shell helper
# All EKS work is CLI-driven. This helper normalises subprocess handling so
# every cell either succeeds explicitly or raises with the full stderr attached.
def sh(cmd, check=True, capture=True, timeout=None, env=None):
    if not ALLOW_MUTATIONS and any(w in cmd for w in ["create", "apply", "delete", "scale"]):
        print(f"[DRY-RUN] {cmd}")
        return ""
    r = subprocess.run(cmd, shell=True, capture_output=capture, text=True, timeout=timeout, env=env)
    if check and r.returncode != 0:
        raise RuntimeError(
            f"Command failed (exit {r.returncode}): {cmd}\\n"
            f"--- stdout ---\\n{r.stdout}\\n"
            f"--- stderr ---\\n{r.stderr}"
        )
    return (r.stdout or "").strip()

def sh_stream(cmd, timeout=None):
    if not ALLOW_MUTATIONS:
        print(f"[DRY-RUN] {cmd}")
        return 0
    r = subprocess.run(cmd, shell=True, timeout=timeout)
    return r.returncode

print("Shell helpers ready.")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Shell helpers ready.


In [16]:
%%jshell

runPy("""
# Block 4b - Verify AWS CLI and eksctl credentials
# Preserve ~/.aws files; exported Colab credentials take precedence.
os.environ['AWS_DEFAULT_REGION']=region; os.environ['AWS_REGION']=region
os.environ.pop('AWS_PROFILE',None); os.environ.pop('AWS_DEFAULT_PROFILE',None)
for command in [f'aws sts get-caller-identity --region {region}',f'eksctl get cluster --region {region}']:
    result=subprocess.run(command,shell=True,capture_output=True,text=True); print(result.stdout or result.stderr)
    if result.returncode: raise RuntimeError(f'Credential check failed: {command}')
print('AWS CLI and eksctl can authenticate. Safe to proceed to Block 5.')
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... {
    "UserId": "455865672536",
    "Account": "455865672536",
    "Arn": "arn:aws:iam::455865672536:root"
}

NAME		REGION		EKSCTL CREATED
eks-lab-demo	us-east-2	True

AWS CLI and eksctl can authenticate. Safe to proceed to Block 5.


## 2. Provision the EKS Cluster

`eksctl create cluster` uses CloudFormation under the hood — you'll see two stacks appear (`eksctl-<name>-cluster`, `eksctl-<name>-nodegroup-*`).

This block is **idempotent**: if a cluster with the same name already exists (from a previous run), we skip creation and reconfigure `kubectl` to point at it.

In [17]:
%%jshell

runPy("""
# Block 4c - Do not create the cluster here.
# Block 5 generates eks_work/cluster.yaml and handles existing or stale stacks.
print("Skipped manual eksctl create. Run Block 5 for idempotent provisioning.")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Skipped manual eksctl create. Run Block 5 for idempotent provisioning.


In [18]:
%%jshell

runPy("""
# Block 6 - EKS Cluster & Nodegroup Provisioning / Reconciliation
import subprocess, yaml
from botocore.exceptions import ClientError


# 1. Helper functions
def get_eks_nodegroup_info(cluster_name, nodegroup_name):
    try:
        response = eks.describe_nodegroup(
            clusterName=cluster_name,
            nodegroupName=nodegroup_name
        )
        return response['nodegroup']

    except ClientError as e:
        if e.response['Error']['Code'] == 'ResourceNotFoundException':
            return None

        print(
            f"Error describing nodegroup {nodegroup_name}: {e}"
        )

        return None


def status():
    try:
        r = eks.describe_cluster(
            name=cluster_name
        )

        return r["cluster"]["status"]

    except ClientError as e:
        if e.response["Error"]["Code"] == "ResourceNotFoundException":
            return None

        raise

    except Exception as e:
        print(
            f"Error getting cluster status: {e}"
        )

        return None


def stale():
    try:
        stacks_response = cfn.list_stacks(
            StackStatusFilter=[
                'CREATE_FAILED',
                'ROLLBACK_FAILED',
                'UPDATE_FAILED',
                'DELETE_FAILED'
            ]
        )

        for stack in stacks_response.get(
            'StackSummaries',
            []
        ):
            if stack['StackName'].startswith(
                f"eksctl-{cluster_name}"
            ):
                return True

        return False

    except Exception as e:
        print(
            f"Warning: Unexpected error checking for stale stacks: {e}"
        )

        return False


# 2. Cleanup existing stale nodegroup stack if present
nodegroup_stack_name = (
    f"eksctl-{cluster_name}-nodegroup-{nodegroup_name}"
)

try:
    cfn.delete_stack(
        StackName=nodegroup_stack_name
    )

    print(
        f"Issued delete request for stack: {nodegroup_stack_name}"
    )

except ClientError as e:
    print(
        f"Stack deletion skip/warning: {e}"
    )


# 3. Check cluster state and reconcile
state = status()


if state is None and stale():

    print(
        'Removing stale eksctl stack...'
    )

    if sh_stream(
        f'eksctl delete cluster --name {cluster_name} '
        f'--region {region} --wait',
        timeout=1800
    ):
        raise RuntimeError(
            'Could not remove stale stack; '
            'delete it in CloudFormation and rerun.'
        )

    state = status()


if state == 'ACTIVE':

    print(
        'Cluster already ACTIVE. '
        'Checking/ensuring nodegroup readiness...'
    )

    ng_info = get_eks_nodegroup_info(
        cluster_name,
        nodegroup_name
    )

    ng_status = (
        ng_info['status']
        if ng_info
        else None
    )

    if ng_status in (
        'ACTIVE',
        'CREATING',
        'UPDATING'
    ):

        print(
            f"Nodegroup '{nodegroup_name}' "
            f"is already in {ng_status} state. "
            f"Proceeding."
        )

    else:

        print(
            f"Nodegroup '{nodegroup_name}' "
            f"status: {ng_status}. "
            f"Reconciling..."
        )

        nodegroup_create_cmd = (
            f"eksctl create nodegroup --cluster {cluster_name} "
            f"--name {nodegroup_name} "
            f"--instance-types {NODE_INSTANCE_TYPE} "
            f"--nodes {NODE_DESIRED} "
            f"--nodes-min {NODE_MIN} "
            f"--nodes-max {NODE_MAX} "
            f"--region {region} "
            f"--managed"
        )

        print(
            f"Executing: {nodegroup_create_cmd}"
        )

        result = subprocess.run(
            nodegroup_create_cmd,
            shell=True,
            capture_output=True,
            text=True,
            timeout=1800
        )

        if result.returncode != 0:

            print(
                f"eksctl create nodegroup failed "
                f"(exit code {result.returncode})."
            )

            print(
                "--- stdout ---\\n" + result.stdout
            )

            print(
                "--- stderr ---\\n" + result.stderr
            )

            try:

                events = cfn.describe_stack_events(
                    StackName=nodegroup_stack_name
                )['StackEvents']

                errors = [
                    e
                    for e in events
                    if (
                        e.get('ResourceStatus') == 'CREATE_FAILED'
                        or
                        'failed'
                        in e.get(
                            'ResourceStatusReason',
                            ''
                        ).lower()
                    )
                ]

                for ev in errors[:5]:

                    print(
                        f"  [{ev['Timestamp'].isoformat()}] "
                        f"{ev['ResourceStatus']} - "
                        f"{ev.get('ResourceStatusReason', '')}"
                    )

            except Exception as ex:

                print(
                    f"Could not retrieve stack events: {ex}"
                )

            raise RuntimeError(
                'Nodegroup reconciliation failed.'
            )

    print(
        'Nodegroup ensured. Updating kubeconfig...'
    )


elif state in (
    'CREATING',
    'UPDATING'
):

    print(
        'Cluster creation/update in progress. '
        'Waiting for ACTIVE state...'
    )

    eks.get_waiter(
        'cluster_active'
    ).wait(
        name=cluster_name
    )

    print(
        'Cluster is now ACTIVE.'
    )


elif state is None:

    manifest = {
        'apiVersion': 'eksctl.io/v1alpha5',
        'kind': 'ClusterConfig',
        'metadata': {
            'name': cluster_name,
            'region': region,
            'version': '1.31'
        },
        'managedNodeGroups': [
            {
                'name': nodegroup_name,
                'instanceType': NODE_INSTANCE_TYPE,
                'minSize': NODE_MIN,
                'maxSize': NODE_MAX,
                'desiredCapacity': NODE_DESIRED,
                'volumeSize': 20
            }
        ],
        'iam': {
            'withOIDC': True
        }
    }

    f = work_dir / 'cluster.yaml'

    f.write_text(
        yaml.safe_dump(manifest)
    )

    if sh_stream(
        f'eksctl create cluster -f {f}',
        timeout=1800
    ):
        raise RuntimeError(
            'Cluster creation failed.'
        )

    print(
        'Cluster created successfully.'
    )


else:

    raise RuntimeError(
        f'Unexpected cluster state: {state}'
    )


# 4. Update kubeconfig and wait for nodes
sh(
    f'aws eks update-kubeconfig '
    f'--name {cluster_name} '
    f'--region {region}'
)

sh(
    'kubectl wait '
    '--for=condition=Ready '
    'nodes '
    '--all '
    '--timeout=600s'
)

print(
    'Cluster and worker nodes are ready.'
)
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Stack deletion skip/warning: An error occurred (ValidationError) when calling the DeleteStack operation: Stack [eksctl-eks-lab-demo-nodegroup-ng-primary] cannot be deleted while TerminationProtection is enabled
Cluster already ACTIVE. Checking/ensuring nodegroup readiness...
Nodegroup 'ng-primary' is already in ACTIVE state. Proceeding.
Nodegroup ensured. Updating kubeconfig...
Cluster and worker nodes are ready.


In [19]:
%%jshell

runPy("""
# import subprocess, os, pathlib, shutil, time

# # 1. Back up the current file (in case you want to restore it later)
# src = pathlib.Path.home() / ".aws" / "config"
# if src.exists():
#     bak = src.with_suffix(f".bak.{int(time.time())}")
#     shutil.copy2(src, bak)
#     print(f"Backed up existing config to {bak}")

# # 2. Rewrite ~/.aws/config with just the region, dropping the malformed sections.
# #    The container-credentials env var (AWS_CONTAINER_CREDENTIALS_RELATIVE_URI)
# #    is what will actually supply credentials to eksctl / aws CLI / boto3.
# src.parent.mkdir(parents=True, exist_ok=True)
# src.write_text("[default]\\nregion = eu-north-1\\n")
# print(f"Rewrote {src}:")
# print(src.read_text())

# # 3. Make sure no residual profile override is set for this kernel
# os.environ.pop("AWS_PROFILE", None)
# os.environ.pop("AWS_DEFAULT_PROFILE", None)
# os.environ["AWS_REGION"] = "eu-north-1"

# # 4. Verify with the aws CLI (same code path eksctl uses)
# r = subprocess.run("aws sts get-caller-identity", shell=True,
#                    capture_output=True, text=True)
# print("\\naws CLI check:")
# print("  stdout:", r.stdout.strip())
# print("  stderr:", r.stderr.strip())
""");


In [20]:
%%jshell

runPy("""
# Block 6 - Wire kubectl at the new cluster
sh(f"aws eks update-kubeconfig --region {region} --name {cluster_name}")
print("kubectl context:", sh("kubectl config current-context"))
print()
print("Cluster info:")
print(sh("kubectl cluster-info"))
print()
print("Nodes:")
print(sh("kubectl get nodes -o wide"))
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... kubectl context: arn:aws:eks:us-east-2:455865672536:cluster/eks-lab-demo

Cluster info:
Kubernetes control plane is running at https://D3E06F87FBBEEC2FD2356663776793DE.gr7.us-east-2.eks.amazonaws.com
CoreDNS is running at https://D3E06F87FBBEEC2FD2356663776793DE.gr7.us-east-2.eks.amazonaws.com/api/v1/namespaces/kube-system/services/kube-dns:dns/proxy

To further debug and diagnose cluster problems, use 'kubectl cluster-info dump'.

Nodes:
NAME                                           STATUS   ROLES    AGE   VERSION                INTERNAL-IP      EXTERNAL-IP    OS-IMAGE                        KERNEL-VERSION                    CONTAINER-RUNTIME
ip-192-168-40-177.us-east-2.compute.internal   Ready    <none>   11m   v1.31.14-eks-f4fc4f1   192.168.40.177   3.21.129.183   Amazon Linux 2023.12.20260918   6.1.186-228.376.amzn2023.x86_64   containerd://2.2.7+unknown


## 2b. UI Checkpoints — Prove EKS and the workloads are up

Everything in this section is **read-only**. Nothing here mutates AWS or the cluster.
The classroom rhythm mirrors Lesson 4:

1. Run **Block 6b** — cluster status card + AWS Console button + inline `kubectl get` tables (nodes, pods, services, deployments, HPA)
2. Click the console button → real EKS console opens in a new tab
3. **Re-run Block 6b any time you want a refreshed view** — no expiry, unlike the MLflow presigned URL

You will come back to Block 6b **five times** in this notebook:

| Checkpoint | When | What Block 6b shows |
|---|---|---|
| 1 | Right after Block 6 (kubectl wired) | Cluster ACTIVE, 1 node, no user pods yet |
| 2 | After Block 10 (v1 deployed) | 2 `inference-v1` pods `Running`, ClusterIP service |
| 3 | After Block 13 (load test + HPA) | Pod count scaled up (2 → N), HPA current CPU % visible |
| 4 | After Block 16 (rolling update) | Pods now show `inference-v2` image |
| 5 | After Block 18 (blue/green cutover) | `green` pods serving; `blue` pods still present until Block 19 |

For the **public HTTP endpoint** (the actual deployed app users can hit),
see **Block 20b** further down after the LoadBalancer is provisioned.


In [21]:
%%jshell

runPy("""
# Block 6b - "Proof EKS is up" cell for students
# Read-only. Re-run any time to refresh cluster + workload state.
# No expiry, no AWS mutations.
import json as _json
import pandas as _pd
from IPython.display import display, Markdown, HTML

# --- 1. Cluster status card ---
try:
    _c = eks.describe_cluster(name=cluster_name)["cluster"]
    _status = _c["status"]
    _endpoint = _c.get("endpoint", "(pending)")
    _version = _c.get("version", "?")
    _arn = _c.get("arn", "?")
except Exception as _e:
    _status, _endpoint, _version, _arn = "NotFound", "-", "-", str(_e)

display(Markdown(f'''
### EKS Cluster
- **Name:**    `{cluster_name}`
- **Status:**  `{_status}` {"✅" if _status == "ACTIVE" else "⏳ (wait for ACTIVE)"}
- **Version:** `{_version}`
- **API endpoint:** `{_endpoint}`
- **ARN:** `{_arn}`
- **Region:** `{region}`
'''))

# --- 2. Clickable AWS Console buttons (deterministic URLs, no expiry) ---
_console_url = (
    f"https://{region}.console.aws.amazon.com/eks/clusters/"
    f"{cluster_name}?region={region}"
)
_cw_url = (
    f"https://{region}.console.aws.amazon.com/cloudwatch/home?region={region}"
    f"#logsV2:log-groups"
)
_btn_dark = (
    "display:inline-block;margin-right:10px;padding:10px 18px;background:#232F3E;"
    "color:#fff;text-decoration:none;border-radius:4px;font-weight:bold;"
)
_btn_gray = (
    "display:inline-block;padding:10px 18px;background:#557;"
    "color:#fff;text-decoration:none;border-radius:4px;font-weight:bold;"
)
_html = (
    '<div style="padding:10px 0;">'
    f'<a href="{_console_url}" target="_blank" style="{_btn_dark}">'
    '▶ Open EKS Console (Cluster page)</a>'
    f'<a href="{_cw_url}" target="_blank" style="{_btn_gray}">'
    '▶ CloudWatch log groups</a>'
    '<div style="margin-top:6px;color:#666;font-size:12px;">'
    'Console links require you to already be signed in to the AWS account.'
    '</div></div>'
)
display(HTML(_html))

# --- 3. Inline kubectl tables (no clicking needed) ---
def _kget_df(args, cols):
    '''Run kubectl get ... -o json and return a small DataFrame.

    cols values may be:
      - a dotted string path (e.g. "metadata.name" or "spec.containers.[0].image")
      - a callable it -> value (use this when a key contains dots, e.g. K8s labels)
    '''
    try:
        raw = sh(f"kubectl get {args} -o json", check=False)
        if not raw or not raw.strip().startswith("{"):
            return None
        items = _json.loads(raw).get("items", [])
        rows = []
        for it in items:
            row = {}
            for c, path in cols.items():
                if callable(path):
                    try:
                        row[c] = path(it)
                    except Exception:
                        row[c] = None
                    continue
                cur = it
                for k in path.split("."):
                    if cur is None:
                        break
                    if k.startswith("[") and k.endswith("]"):
                        idx = int(k[1:-1])
                        cur = cur[idx] if isinstance(cur, list) and abs(idx) <= len(cur) else None
                    else:
                        cur = cur.get(k) if isinstance(cur, dict) else None
                row[c] = cur
            rows.append(row)
        return _pd.DataFrame(rows)
    except Exception as e:
        return _pd.DataFrame([{"error": str(e)}])

def _label(key):
    '''Callable path helper for label keys that contain dots (e.g. 'node.kubernetes.io/instance-type').'''
    return lambda it: (it.get("metadata", {}).get("labels") or {}).get(key)

def _ready_condition(it):
    '''Find the 'Ready' condition status explicitly (don't rely on ordering).'''
    for cond in (it.get("status", {}).get("conditions") or []):
        if cond.get("type") == "Ready":
            return cond.get("status")
    return "?"

display(Markdown("### Nodes"))
_nodes = _kget_df("nodes", {
    "name":    "metadata.name",
    "ready":   _ready_condition,
    "version": "status.nodeInfo.kubeletVersion",
    "type":    _label("node.kubernetes.io/instance-type"),
})
display(_nodes if _nodes is not None else Markdown("_(no nodes visible)_"))

display(Markdown(f"### Pods in `{namespace}`"))
_pods = _kget_df(f"pods -n {namespace}", {
    "name":     "metadata.name",
    "phase":    "status.phase",
    "ready":    "status.containerStatuses.[0].ready",
    "restarts": "status.containerStatuses.[0].restartCount",
    "image":    "spec.containers.[0].image",
    "args":     "spec.containers.[0].args",
})
if _pods is not None and not _pods.empty:
    display(_pods)
else:
    display(Markdown("_(no pods in this namespace yet — Block 9+ will create them)_"))

display(Markdown(f"### Services in `{namespace}`"))
_svcs = _kget_df(f"svc -n {namespace}", {
    "name":       "metadata.name",
    "type":       "spec.type",
    "clusterIP":  "spec.clusterIP",
    "port":       "spec.ports.[0].port",
    "selector":   "spec.selector",
    "external":   "status.loadBalancer.ingress.[0].hostname",
})
if _svcs is not None and not _svcs.empty:
    display(_svcs)
else:
    display(Markdown("_(no services yet — Block 10 will create one)_"))

display(Markdown(f"### Deployments in `{namespace}`"))
_deps = _kget_df(f"deployments -n {namespace}", {
    "name":      "metadata.name",
    "replicas":  "spec.replicas",
    "available": "status.availableReplicas",
    "image":     "spec.template.spec.containers.[0].image",
    "strategy":  "spec.strategy.type",
})
if _deps is not None and not _deps.empty:
    display(_deps)
else:
    display(Markdown("_(no deployments yet — Block 9 will create one)_"))

display(Markdown(f"### HorizontalPodAutoscalers in `{namespace}`"))
_hpas = _kget_df(f"hpa -n {namespace}", {
    "name":     "metadata.name",
    "target":   "spec.scaleTargetRef.name",
    "min":      "spec.minReplicas",
    "max":      "spec.maxReplicas",
    "current":  "status.currentReplicas",
    "cpu_pct":  "status.currentCPUUtilizationPercentage",
})
if _hpas is not None and not _hpas.empty:
    display(_hpas)
else:
    display(Markdown("_(no HPA yet — Block 12 will create one)_"))
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... <IPython.core.display.Markdown object>
<IPython.core.display.HTML object>
<IPython.core.display.Markdown object>
                                           name  ...       type
0  ip-192-168-40-177.us-east-2.compute.internal  ...  t3.medium

[1 rows x 4 columns]
<IPython.core.display.Markdown object>
                                name  ...                                        args
0  inference-server-6746dddc97-tx4f9  ...  [-text=inference-server-v1, -listen=:5678]
1  inference-server-6746dddc97-vr99w  ...  [-text=inference-server-v1, -listen=:5678]

[2 rows x 6 columns]
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>
               name  replicas available                    image       strategy
0  inference-server         2      None  hashicorp/http-echo:1.0  RollingUpdate
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown obj

## 3. Deploy v1 — Namespace, Config, Secret, Deployment, Service

Every K8s object is declarative YAML. We generate the manifests, write them to disk (audit trail), and apply them. `kubectl apply` is idempotent — re-running finds no diff and no-ops.

**What we build:**
1. **Namespace** — isolate the lab from anything else in the cluster
2. **ConfigMap** — non-sensitive config (model name, batch size)
3. **Secret** — sensitive data (API keys) — base64-encoded, not encrypted-at-rest by default
4. **Deployment** — 2 replicas of the inference server with readiness/liveness probes, ConfigMap + Secret env vars, and a `RollingUpdate` strategy
5. **Service** — stable ClusterIP DNS that load-balances to healthy pods

In [22]:
%%jshell

runPy("""
# Block 7 - Namespace
ns_yaml = {
    "apiVersion": "v1",
    "kind":       "Namespace",
    "metadata": {"name": namespace, "labels": {"purpose": "lesson5-lab"}},
}
p = work_dir / "namespace.yaml"
with open(p, "w") as f: yaml.safe_dump(ns_yaml, f)
print(sh(f"kubectl apply -f {p}"))
print(sh(f"kubectl get ns {namespace}"))
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... namespace/ml-lab unchanged
NAME     STATUS   AGE
ml-lab   Active   3m42s


In [24]:
%%jshell

runPy("""
# Block 8 - ConfigMap + Secret
cm_yaml = {
    "apiVersion": "v1", "kind": "ConfigMap",
    "metadata": {"name": "model-config", "namespace": namespace},
    "data": {
        "MODEL_NAME":    "sentiment-classifier",
        "MODEL_VERSION": "v1.2",
        "BATCH_SIZE":    "32",
        "MAX_SEQ_LEN":   "128",
    },
}
p = work_dir / "configmap.yaml"
with open(p, "w") as f: yaml.safe_dump(cm_yaml, f)
print(sh(f"kubectl apply -f {p}"))

# Secret via create - the imperative --dry-run=client -o yaml | apply -f - pattern
# is the idiomatic way to make Secret creation both declarative and idempotent.
sh(
    f'''kubectl -n {namespace} create secret generic api-keys \
        --from-literal=OPENAI_KEY=sk-fake-demo-openai-key \
        --from-literal=HF_TOKEN=hf_fake_demo_token \
        --dry-run=client -o yaml | kubectl apply -f -'''
)
print()
print(sh(f"kubectl -n {namespace} describe configmap model-config"))
print(sh(f"kubectl -n {namespace} get secret api-keys"))
print("\\n(Secret values are never shown by `kubectl get`. They exist, base64-encoded, on the API server's etcd.)")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... configmap/model-config created

Name:         model-config
Namespace:    ml-lab
Labels:       <none>
Annotations:  <none>

Data
====
BATCH_SIZE:
----
32

MAX_SEQ_LEN:
----
128

MODEL_NAME:
----
sentiment-classifier

MODEL_VERSION:
----
v1.2


BinaryData
====

Events:  <none>
NAME       TYPE     DATA   AGE
api-keys   Opaque   2      2s

(Secret values are never shown by `kubectl get`. They exist, base64-encoded, on the API server's etcd.)


In [25]:
%%jshell

runPy("""
# Block 9 - Deployment v1
# Key production patterns:
#   - readinessProbe with initialDelaySeconds >= real model load time
#   - resource requests + limits (HPA and cluster autoscaler both use these)
#   - env from ConfigMap + Secret (config decoupled from image)
#   - RollingUpdate maxUnavailable=0 (never lose serving capacity mid-deploy)

def build_deployment(text, replicas=2):
    return {
        "apiVersion": "apps/v1", "kind": "Deployment",
        "metadata": {"name": "inference-server", "namespace": namespace,
                     "labels": {"app": "inference", "version": "v1"}},
        "spec": {
            "replicas": replicas,
            "selector": {"matchLabels": {"app": "inference"}},
            "strategy": {"type": "RollingUpdate",
                         "rollingUpdate": {"maxSurge": 1, "maxUnavailable": 0}},
            "template": {
                "metadata": {"labels": {"app": "inference", "version": "v1"}},
                "spec": {"containers": [{
                    "name":  "model-server",
                    "image": APP_IMAGE,
                    "args":  [f"-text={text}", f"-listen=:{APP_PORT}"],
                    "ports": [{"containerPort": APP_PORT, "name": "http"}],
                    "env": [
                        {"name": "MODEL_NAME",
                         "valueFrom": {"configMapKeyRef": {"name": "model-config", "key": "MODEL_NAME"}}},
                        {"name": "BATCH_SIZE",
                         "valueFrom": {"configMapKeyRef": {"name": "model-config", "key": "BATCH_SIZE"}}},
                        {"name": "OPENAI_KEY",
                         "valueFrom": {"secretKeyRef": {"name": "api-keys", "key": "OPENAI_KEY"}}},
                    ],
                    "resources": {
                        "requests": {"cpu": "100m", "memory": "64Mi"},
                        "limits":   {"cpu": "500m", "memory": "256Mi"},
                    },
                    "readinessProbe": {
                        "httpGet": {"path": "/", "port": APP_PORT},
                        "initialDelaySeconds": 3, "periodSeconds": 5,
                    },
                    "livenessProbe": {
                        "httpGet": {"path": "/", "port": APP_PORT},
                        "initialDelaySeconds": 15, "periodSeconds": 20,
                    },
                }]},
            },
        },
    }

dep = build_deployment(APP_TEXT_V1)
p = work_dir / "deployment.yaml"
with open(p, "w") as f: yaml.safe_dump(dep, f)
print(sh(f"kubectl apply -f {p}"))
print()
print("Waiting for rollout...")
sh_stream(f"kubectl -n {namespace} rollout status deployment/inference-server --timeout=180s")
print()
print(sh(f"kubectl -n {namespace} get pods -o wide"))
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... deployment.apps/inference-server unchanged

Waiting for rollout...
deployment "inference-server" successfully rolled out

NAME                                READY   STATUS    RESTARTS   AGE    IP               NODE                                           NOMINATED NODE   READINESS GATES
inference-server-6746dddc97-tx4f9   1/1     Running   0          5m1s   192.168.33.165   ip-192-168-40-177.us-east-2.compute.internal   <none>           <none>
inference-server-6746dddc97-vr99w   1/1     Running   0          5m1s   192.168.34.225   ip-192-168-40-177.us-east-2.compute.internal   <none>           <none>


In [33]:
%%jshell

runPy("""
# Block 10 - Service (ClusterIP) + Targeted Smoke Test

endpoints = sh(
    'kubectl -n kube-system get endpoints aws-load-balancer-webhook-service -o jsonpath="{.subsets[*].addresses[*].ip}"',
    check=False
)

if not endpoints:
    print('Removing stale ALB mutating webhook with no endpoints...')
    sh('kubectl delete mutatingwebhookconfiguration aws-load-balancer-webhook --ignore-not-found', check=False)

# 1. Define Service matching the Block 9 app selector
svc = {
    'apiVersion': 'v1',
    'kind': 'Service',
    'metadata': {
        'name': 'inference-svc',
        'namespace': namespace
    },
    'spec': {
        'type': 'ClusterIP',
        'selector': {
            'app': 'inference'
        },
        'ports': [
            {
                'port': 80,
                'targetPort': APP_PORT,
                'protocol': 'TCP'
            }
        ]
    }
}

f = work_dir / 'service.yaml'
f.write_text(yaml.safe_dump(svc))
print(sh(f'kubectl apply -f {f}'))

# 2. Wait for Service endpoints to populate
print('Waiting for inference-svc endpoints to register...')
sh(f'kubectl -n {namespace} wait --for=jsonpath="{{.subsets[0].addresses[0].ip}}" endpoints/inference-svc --timeout=60s')

# 3. Perform curl smoke test using evaluated {namespace}
target_url = f'http://inference-svc.{namespace}.svc.cluster.local'

resp = sh(
    f'kubectl -n {namespace} run curl-test '
    '--rm -i --restart=Never '
    '--image=curlimages/curl:8.5.0 '
    f'--command -- curl -sS {target_url}',
    check=False
)

# Extract response body lines, filtering out kubectl pod deletion output
response_lines = [l.strip() for l in resp.splitlines() if not l.startswith('pod "curl-test"')]
actual_text = "\\n".join(response_lines)

expected_text = globals().get('APP_TEXT_V1', 'inference-server-v1')

assert expected_text in actual_text, (
    f"Expected {expected_text!r} in response, but got:\\n{resp}"
)

print(f"Service verified! Target deployment is serving: {expected_text}")
""");

->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Removing stale ALB mutating webhook with no endpoints...
service/inference-svc unchanged
Waiting for inference-svc endpoints to register...
Service verified! Target deployment is serving: inference-server-v1


## 4. Autoscaling — metrics-server + HPA + load test

The HPA needs a metrics source. EKS clusters ship without `metrics-server`, so we install it first. Then we create an HPA that scales `inference-server` from 2 → 6 replicas based on CPU utilization, and drive real load with a background pod.

**Expected outcome:** the HPA will spin up 3–4 pods during load, then scale back down over the next 5 minutes once the load stops (default cool-down).

In [37]:
%%jshell

runPy("""
import subprocess, time, json

def _managed_by():
    r = subprocess.run(
        r"kubectl -n kube-system get deployment metrics-server -o jsonpath='{.metadata.labels.app\\.kubernetes\\.io/managed-by}' 2>/dev/null",
        shell=True, capture_output=True, text=True,
    )
    return (r.stdout or "").strip()

def _metrics_api_ready():
    r = subprocess.run("kubectl top nodes", shell=True, capture_output=True, text=True)
    return r.returncode == 0 and "CPU(cores)" in r.stdout

managed = _managed_by()

if managed == "EKS":
    print(f"Detected EKS-managed metrics-server (managed-by={managed}). Skipping install.")

    dep_labels = subprocess.run(
        "kubectl -n kube-system get deployment metrics-server -o jsonpath='{.spec.selector.matchLabels}'",
        shell=True, capture_output=True, text=True,
    ).stdout.strip()
    svc_labels = subprocess.run(
        "kubectl -n kube-system get svc metrics-server -o jsonpath='{.spec.selector}'",
        shell=True, capture_output=True, text=True,
    ).stdout.strip()
    print(f"Deployment selector: {dep_labels}")
    print(f"Service selector:    {svc_labels}")

    if svc_labels != dep_labels:
        print("Selector mismatch -> REPLACING Service selector with Deployment's labels...")

        patch_payload = json.dumps([
            {
                "op": "replace",
                "path": "/spec/selector",
                "value": {
                    "app.kubernetes.io/instance": "metrics-server",
                    "app.kubernetes.io/name": "metrics-server"
                }
            }
        ])

        sh(f"kubectl -n kube-system patch svc metrics-server --type=json -p='{patch_payload}'")
        time.sleep(3)
        print(sh("kubectl -n kube-system get endpoints metrics-server"))
    else:
        print("Service selector already matches. No repair needed.")

else:
    if managed:
        print(f"metrics-server present but managed-by={managed!r} (not EKS). Skipping install.")
    else:
        print("No metrics-server found. Installing upstream community manifest...")
        print(sh(
            "kubectl apply -f "
            "https://github.com/kubernetes-sigs/metrics-server/releases/latest/download/components.yaml"
        ))
        print("\\nWaiting for metrics-server rollout (up to 90s)...")
        sh_stream("kubectl -n kube-system rollout status deployment/metrics-server --timeout=90s")

print("\\nWaiting for Metrics API to serve samples (up to 90s)...")
for i in range(9):
    if _metrics_api_ready():
        print(f"Metrics API ready after {i*10}s.")
        break
    time.sleep(10)
else:
    raise RuntimeError(
        "Metrics API did not become ready in 90s. Diagnose with:\\n"
        "  kubectl get apiservice v1beta1.metrics.k8s.io -o wide\\n"
        "  kubectl -n kube-system get endpoints metrics-server -o yaml\\n"
        "  kubectl -n kube-system get svc metrics-server -o yaml\\n"
        "  kubectl -n kube-system logs -l app.kubernetes.io/name=metrics-server --tail=40"
    )

print()
print(sh("kubectl -n kube-system get deployment metrics-server"))
print(sh("kubectl top nodes"))
""");

->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Detected EKS-managed metrics-server (managed-by=EKS). Skipping install.
Deployment selector: {"app.kubernetes.io/instance":"metrics-server","app.kubernetes.io/name":"metrics-server"}
Service selector:    {"app.kubernetes.io/instance":"metrics-server","app.kubernetes.io/name":"metrics-server"}
Service selector already matches. No repair needed.

Waiting for Metrics API to serve samples (up to 90s)...
Metrics API ready after 0s.

NAME             READY   UP-TO-DATE   AVAILABLE   AGE
metrics-server   2/2     2            2           23m
NAME                                           CPU(cores)   CPU(%)   MEMORY(bytes)   MEMORY(%)   
ip-192-168-40-177.us-east-2.compute.internal   43m          2%       558Mi           16%


In [38]:
%%jshell

runPy("""
# Block 12 - HorizontalPodAutoscaler
hpa_yaml = {
    "apiVersion": "autoscaling/v2", "kind": "HorizontalPodAutoscaler",
    "metadata": {"name": "inference-hpa", "namespace": namespace},
    "spec": {
        "scaleTargetRef": {"apiVersion": "apps/v1", "kind": "Deployment",
                           "name": "inference-server"},
        "minReplicas": 2, "maxReplicas": 6,
        "metrics": [{
            "type": "Resource",
            "resource": {"name": "cpu", "target": {"type": "Utilization",
                                                    "averageUtilization": 50}},
        }],
    },
}
p = work_dir / "hpa.yaml"
with open(p, "w") as f: yaml.safe_dump(hpa_yaml, f)
print(sh(f"kubectl apply -f {p}"))
print()
# HPA needs 30-60s for metrics-server to publish its first sample.
# The <unknown>/50% output is normal for the first minute.
print(sh(f"kubectl -n {namespace} get hpa"))
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... horizontalpodautoscaler.autoscaling/inference-hpa created

NAME            REFERENCE                     TARGETS              MINPODS   MAXPODS   REPLICAS   AGE
inference-hpa   Deployment/inference-server   cpu: <unknown>/50%   2         6         0          1s


In [40]:
%%jshell

runPy("""
# Block 13 - Generate real load
# We run a background pod that hammers the Service with `wget`. Once it exits,
# HPA will scale replicas back down over the next ~5 minutes.
sh(f"kubectl -n {namespace} delete pod load-generator --ignore-not-found", check=False)

load_pod_yaml = {
    "apiVersion": "v1", "kind": "Pod",
    "metadata": {"name": "load-generator", "namespace": namespace},
    "spec": {
        "restartPolicy": "Never",
        "containers": [{
            "name": "busybox", "image": "busybox:1.36",
            "command": ["/bin/sh", "-c",
                        "end=$(($(date +%s)+120)); "
                        "while [ $(date +%s) -lt $end ]; do "
                        "  wget -q -O- http://inference-svc.ml-lab.svc.cluster.local >/dev/null; "
                        "done"],
        }],
    },
}
p = work_dir / "load.yaml"
with open(p, "w") as f: yaml.safe_dump(load_pod_yaml, f)
print(sh(f"kubectl apply -f {p}"))
print("Load generator running for 120s in the background.")
print()
print("Watching HPA + pod count for 3 minutes (60s intervals):")
for i in range(3):
    time.sleep(60)
    print(f"\\n--- t+{(i+1)*60}s ---")
    print(sh(f"kubectl -n {namespace} get hpa inference-hpa"))
    pods = sh(f"kubectl -n {namespace} get pods -l app=inference --no-headers | wc -l")
    print(f"inference pod count: {pods.strip()}")

print("\\n>>> Re-run Block 6b every 30s to watch HPA scale replicas up. See the HPA table's current/cpu_pct columns.")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... pod/load-generator created
Load generator running for 120s in the background.

Watching HPA + pod count for 3 minutes (60s intervals):

--- t+60s ---
NAME            REFERENCE                     TARGETS        MINPODS   MAXPODS   REPLICAS   AGE
inference-hpa   Deployment/inference-server   cpu: 44%/50%   2         6         2          78s
inference pod count: 2

--- t+120s ---
NAME            REFERENCE                     TARGETS        MINPODS   MAXPODS   REPLICAS   AGE
inference-hpa   Deployment/inference-server   cpu: 45%/50%   2         6         2          2m20s
inference pod count: 2

--- t+180s ---
NAME            REFERENCE                     TARGETS       MINPODS   MAXPODS   REPLICAS   AGE
inference-hpa   Deployment/inference-server   cpu: 1%/50%   2         6         2          3m22s
inference pod count: 2

>>> Re-run Block 6b every 30s to watch HPA scale replicas up. See the HPA table's current/cpu_pct colu

In [41]:
%%jshell

runPy("""
# Block 14 - Cleanup the load generator (HPA scale-down happens on its own timer)
sh(f"kubectl -n {namespace} delete pod load-generator --ignore-not-found")
print("Load generator removed. HPA will scale replicas down over the next ~5 min.")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Load generator removed. HPA will scale replicas down over the next ~5 min.


## 5. Rolling update v1 → v2

Change the container's response text, re-apply. Because `maxUnavailable=0` in the deployment strategy, K8s creates a new pod before terminating an old one — no downtime.

We verify by hitting the Service while the rollout is in progress and confirming responses transition cleanly.

In [45]:
%%jshell

runPy("""
# Block 15 - Rolling update by patch
# We patch the deployment's container args directly - kubectl records this as a
# new ReplicaSet, so `kubectl rollout undo` can revert if needed.
sh(
    f'kubectl -n {namespace} patch deployment inference-server '
    f'--type=json '
    f\"\"\"-p='[{{"op":"replace","path":"/spec/template/spec/containers/0/args","value":["-text={APP_TEXT_V2}","-listen=:{APP_PORT}"]}}]'\"\"\"
)

print("Rollout starting...")
sh_stream(f"kubectl -n {namespace} rollout status deployment/inference-server --timeout=180s")
print()
print("Rollout history:")
print(sh(f"kubectl -n {namespace} rollout history deployment/inference-server"))

print("\\n>>> Re-run Block 6b to see pods image column change from v1 to v2 with zero downtime.")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Rollout starting...
Waiting for deployment "inference-server" rollout to finish: 1 out of 2 new replicas have been updated...
Waiting for deployment "inference-server" rollout to finish: 1 out of 2 new replicas have been updated...
Waiting for deployment "inference-server" rollout to finish: 1 out of 2 new replicas have been updated...
Waiting for deployment "inference-server" rollout to finish: 1 old replicas are pending termination...
Waiting for deployment "inference-server" rollout to finish: 1 old replicas are pending termination...
deployment "inference-server" successfully rolled out

Rollout history:
deployment.apps/inference-server 
REVISION  CHANGE-CAUSE
1         <none>
2         <none>

>>> Re-run Block 6b to see pods image column change from v1 to v2 with zero downtime.


In [52]:
%%jshell

runPy("""
# 1. Update deployment args via patch to avoid kubectl flag parsing issues
print("Updating deployment/inference-server to v3...")
patch_json = f'{{"spec":{{"template":{{"spec":{{"containers":[{{"name":"model-server","args":["-text={APP_TEXT_V3_BG}","-listen=:{APP_PORT}"]}}]}}}}}}}}'
print(sh(f"kubectl -n {namespace} patch deployment/inference-server --type=strategic -p '{patch_json}'"))

# 2. Wait for rollout completion
print("\\nWaiting for v3 rollout to complete...")
sh_stream(f"kubectl -n {namespace} rollout status deployment/inference-server --timeout=90s")

# 3. Dynamically inspect active pod names serving traffic
print("\\nActive v3 Pods:")
pod_names = sh(
    f"kubectl -n {namespace} get pods -l app=inference -o jsonpath='{{.items[*].metadata.name}}'"
).split()

for pod in pod_names:
    print(f" - {pod}")

# 4. Perform curl smoke test against Service
print("\\nTesting Service response:")
resp = sh(
    f"kubectl -n {namespace} run curl-v3 --rm -i --restart=Never "
    f"--image=curlimages/curl:8.5.0 --command -- "
    f"curl -sS http://inference-svc.{namespace}.svc.cluster.local",
    check=False
)

response_lines = [l.strip() for l in resp.splitlines() if not l.startswith('pod "curl-v3"')]
actual_text = "\\n".join(response_lines)

expected_text = globals().get('APP_TEXT_V3_BG', 'inference-server-v3-green')

assert expected_text in actual_text, (
    f"Expected '{expected_text}' in response, but got:\\n{resp}"
)

print(f"\\nv3 is successfully serving: {actual_text}")
print(f"To roll back: kubectl -n {namespace} rollout undo deployment/inference-server")
""");

->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Updating deployment/inference-server to v3...
deployment.apps/inference-server patched

Waiting for v3 rollout to complete...
Waiting for deployment "inference-server" rollout to finish: 1 out of 2 new replicas have been updated...
Waiting for deployment "inference-server" rollout to finish: 1 out of 2 new replicas have been updated...
Waiting for deployment "inference-server" rollout to finish: 1 out of 2 new replicas have been updated...
Waiting for deployment "inference-server" rollout to finish: 1 old replicas are pending termination...
Waiting for deployment "inference-server" rollout to finish: 1 old replicas are pending termination...
deployment "inference-server" successfully rolled out

Active v3 Pods:
 - inference-server-5857d8b77c-njnkh
 - inference-server-5857d8b77c-zmkql

Testing Service response:

v3 is successfully serving: inference-server-v3-green
To roll back: kubectl -n ml-lab rollout undo deployment

## 6. Blue/Green cutover to v3

Rolling updates are safe but mix v1 + v2 pods during the transition. Blue/green is different: the **entire new version is deployed alongside the old one**, then a single Service selector change switches all traffic instantly. Instant rollback = flip the selector back.

**Pattern:**
- **Blue** = current live Deployment (`inference-server` running v2)
- **Green** = new Deployment (`inference-server-green` running v3)
- Both exist simultaneously. Service selector controls which one gets traffic.

**Cost note:** while both exist, you're running 2× replicas. Keep the overlap short.

In [53]:
%%jshell

runPy("""
# Block 17 - Create the GREEN Deployment (v3) alongside the BLUE (v2)
green_dep = build_deployment(APP_TEXT_V3_BG)
# Rename it and re-label the pods so the Service can select just one.
green_dep["metadata"]["name"]                              = "inference-server-green"
green_dep["metadata"]["labels"]["color"]                   = "green"
green_dep["metadata"]["labels"]["version"]                 = "v3"
green_dep["spec"]["selector"]["matchLabels"]               = {"app": "inference", "color": "green"}
green_dep["spec"]["template"]["metadata"]["labels"]        = {"app": "inference", "color": "green", "version": "v3"}

p = work_dir / "deployment-green.yaml"
with open(p, "w") as f: yaml.safe_dump(green_dep, f)
print(sh(f"kubectl apply -f {p}"))
print()
print("Waiting for green rollout...")
sh_stream(f"kubectl -n {namespace} rollout status deployment/inference-server-green --timeout=180s")
print()
print("Both blue (v2) and green (v3) are now running:")
print(sh(f"kubectl -n {namespace} get pods -o wide -l app=inference"))
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... deployment.apps/inference-server-green created

Waiting for green rollout...
Waiting for deployment "inference-server-green" rollout to finish: 0 of 2 updated replicas are available...
Waiting for deployment "inference-server-green" rollout to finish: 1 of 2 updated replicas are available...
deployment "inference-server-green" successfully rolled out

Both blue (v2) and green (v3) are now running:
NAME                                     READY   STATUS    RESTARTS   AGE   IP               NODE                                           NOMINATED NODE   READINESS GATES
inference-server-5857d8b77c-njnkh        1/1     Running   0          53s   192.168.33.230   ip-192-168-40-177.us-east-2.compute.internal   <none>           <none>
inference-server-5857d8b77c-zmkql        1/1     Running   0          48s   192.168.37.9     ip-192-168-40-177.us-east-2.compute.internal   <none>           <none>
inference-server-green-96c64df

In [59]:
%%jshell

runPy("""
# Block 18 - Cut the Service selector to green (this is the blue/green cutover)
patch_json = '{"spec":{"selector":{"app":"inference","color":"green"}}}'
sh(f"kubectl -n {namespace} patch service inference-svc --type=merge -p='{patch_json}'")
print("Service selector updated: traffic now goes to green (v3).")

# Fetch active green pod names dynamically
print("\\nTarget Green Pods:")
green_pods = sh(
    f"kubectl -n {namespace} get pods -l app=inference,color=green -o jsonpath='{{.items[*].metadata.name}}'"
).split()
for pod in green_pods:
    print(f" - {pod}")

# Verify - Service should now return v3 text
time.sleep(2)
resp = sh(
    f"kubectl -n {namespace} run curl-v3 --rm -i --restart=Never "
    f"--image=curlimages/curl:8.5.0 --command -- "
    f"curl -sS http://inference-svc.{namespace}.svc.cluster.local",
    check=False
)

# Filter out kubectl cleanup lines
response_lines = [l.strip() for l in resp.splitlines() if not l.startswith('pod "curl-v3"')]
actual_text = "\\n".join(response_lines)

expected_text = globals().get('APP_TEXT_V3_BG', 'inference-server-v3-green')

print(f"\\nResponse after cutover: {actual_text}")
assert expected_text in actual_text, f"Expected '{expected_text}' in response, got: {resp}"

print("\\nCutover successful. Blue (v2) is still running but receives no traffic.")
print("To ROLLBACK instantly:")

rollback_cmd = f"kubectl -n {namespace} patch service inference-svc --type=merge -p='" + '{"spec":{"selector":{"app":"inference","color":"blue"}}}' + "'"
print(f"  {rollback_cmd}")

print("\\n>>> Re-run Block 6b to see the service selector now includes color=green. Blue+green pods coexist until Block 19.")
""");

->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Service selector updated: traffic now goes to green (v3).

Target Green Pods:
 - inference-server-green-96c64df9c-c4dkk
 - inference-server-green-96c64df9c-tvcbf

Response after cutover: inference-server-v3-green

Cutover successful. Blue (v2) is still running but receives no traffic.
To ROLLBACK instantly:
  kubectl -n ml-lab patch service inference-svc --type=merge -p='{"spec":{"selector":{"app":"inference","color":"blue"}}}'

>>> Re-run Block 6b to see the service selector now includes color=green. Blue+green pods coexist until Block 19.


In [60]:
%%jshell

runPy("""
# Block 19 - Retire blue once green is confirmed healthy in production
# In a real deployment you'd wait 10-30 min while monitoring error rates before
# doing this. For the lab we do it immediately.
sh(f"kubectl -n {namespace} delete deployment inference-server --ignore-not-found")
print("Blue (v2) retired. Only green (v3) remains.")
print()
print(sh(f"kubectl -n {namespace} get deployment -l app=inference"))
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Blue (v2) retired. Only green (v3) remains.

NAME                     READY   UP-TO-DATE   AVAILABLE   AGE
inference-server-green   2/2     2            2           16m


## 7. Expose Externally — LoadBalancer Service

A `Service` of type `LoadBalancer` on EKS provisions an AWS Network Load Balancer (NLB). This is the simplest way to get a public endpoint. For production with multiple services, use an Ingress controller (ALB or NGINX) to share one LB.

**Cost:** NLB is ~$0.0225/hr + a per-LCU charge for traffic (negligible for demo).

In [62]:
%%jshell

runPy("""
# Block 19b - Install AWS Load Balancer Controller (one-time per cluster)
# EKS clusters do NOT ship with the AWS Load Balancer Controller. Without it,
# LoadBalancer-type Services with `aws-load-balancer-*` annotations sit at
# <pending> forever because nothing is watching them. Block 20 needs this.
#
# What this cell does:
#   1. Downloads the controller's required IAM policy
#   2. Creates that IAM policy (idempotent)
#   3. Creates an IRSA-bound ServiceAccount (eksctl handles OIDC + role trust)
#   4. Installs Helm if missing
#   5. Deploys the controller via the eks-charts Helm repo
#
# Runtime: ~3 minutes. Only needs to run once per cluster lifetime.
import subprocess, os, urllib.request

_alb_ver = "v2.7.2"
_policy_url = (
    f"https://raw.githubusercontent.com/kubernetes-sigs/"
    f"aws-load-balancer-controller/{_alb_ver}/docs/install/iam_policy.json"
)

# 1. Fetch the IAM policy JSON
_policy_path = "/tmp/alb_iam_policy.json"
print(f"Downloading IAM policy from {_alb_ver}...")
urllib.request.urlretrieve(_policy_url, _policy_path)
print(f"Saved to {_policy_path}")

# 2. Create the IAM policy (skip cleanly if it already exists)
_policy_name = "AWSLoadBalancerControllerIAMPolicy"
_policy_arn = f"arn:aws:iam::{account_id}:policy/{_policy_name}"
try:
    with open(_policy_path) as _f:
        iam.create_policy(PolicyName=_policy_name, PolicyDocument=_f.read())
    print(f"Created IAM policy: {_policy_arn}")
except iam.exceptions.EntityAlreadyExistsException:
    print(f"IAM policy already exists: {_policy_arn}")

# 3. Create the IRSA ServiceAccount (creates OIDC provider on first run)
print("\\nCreating IRSA service account (this creates the OIDC provider too)...")
sh_stream(
    f"eksctl create iamserviceaccount "
    f"--cluster={cluster_name} "
    f"--region={region} "
    f"--namespace=kube-system "
    f"--name=aws-load-balancer-controller "
    f"--attach-policy-arn={_policy_arn} "
    f"--approve "
    f"--override-existing-serviceaccounts",
    timeout=600,
)

# 4. Install Helm if missing
if not shutil.which("helm"):
    print("\\nInstalling helm...")
    subprocess.run(
        "curl -fsSL https://raw.githubusercontent.com/helm/helm/main/scripts/get-helm-3 "
        "-o /tmp/get_helm.sh && chmod +x /tmp/get_helm.sh && "
        "HELM_INSTALL_DIR=$HOME/.local/bin USE_SUDO=false /tmp/get_helm.sh",
        shell=True, check=True,
    )
    os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ.get("PATH", "")
    print("helm installed to ~/.local/bin/")

# 5. Discover the VPC ID (the controller needs it explicitly)
_vpc_id = eks.describe_cluster(name=cluster_name)["cluster"]["resourcesVpcConfig"]["vpcId"]
print(f"\\nVPC ID: {_vpc_id}")

# 6. Add the Helm repo and install the controller
print("\\nAdding eks-charts Helm repo...")
sh("helm repo add eks https://aws.github.io/eks-charts")
sh("helm repo update eks")

print("\\nInstalling aws-load-balancer-controller via Helm...")
sh_stream(
    f"helm upgrade --install aws-load-balancer-controller eks/aws-load-balancer-controller "
    f"-n kube-system "
    f"--set clusterName={cluster_name} "
    f"--set serviceAccount.create=false "
    f"--set serviceAccount.name=aws-load-balancer-controller "
    f"--set region={region} "
    f"--set vpcId={_vpc_id} "
    f"--wait",
    timeout=600,
)

# 7. Verify controller is running
print("\\n=== Controller pods ===")
print(sh("kubectl get pods -n kube-system -l app.kubernetes.io/name=aws-load-balancer-controller"))
print("\\n=== Controller version ===")
print(sh("kubectl get deployment aws-load-balancer-controller -n kube-system "
         "-o jsonpath='{.spec.template.spec.containers[0].image}'"))
print("\\n\\nController ready. Block 20 will now be able to provision an NLB.")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Downloading IAM policy from v2.7.2...
Saved to /tmp/alb_iam_policy.json
IAM policy already exists: arn:aws:iam::455865672536:policy/AWSLoadBalancerControllerIAMPolicy

Creating IRSA service account (this creates the OIDC provider too)...
2026-09-30 07:40:18 [ℹ]  1 iamserviceaccount (kube-system/aws-load-balancer-controller) was included (based on the include/exclude rules)
2026-09-30 07:40:18 [!]  metadata of serviceaccounts that exist in Kubernetes will be updated, as --override-existing-serviceaccounts was set
2026-09-30 07:40:18 [ℹ]  1 task: { 
    2 sequential sub-tasks: { 
        create IAM role for serviceaccount "kube-system/aws-load-balancer-controller",
        create serviceaccount "kube-system/aws-load-balancer-controller",
    } }2026-09-30 07:40:18 [ℹ]  building iamserviceaccount stack "eksctl-eks-lab-demo-addon-iamserviceaccount-kube-system-aws-load-balancer-controller"
2026-09-30 07:40:18 [ℹ]  deploying

In [63]:
%%jshell

runPy("""
# Fix: add missing elasticloadbalancing:Describe* actions to the controller's IAM policy
import json
import subprocess # Added for explicit subprocess calls if sh_stream masks output

_policy_name = "AWSLoadBalancerControllerIAMPolicy"
_policy_arn = f"arn:aws:iam::{account_id}:policy/{_policy_name}"

# Get the current default version of the policy
_pv = iam.get_policy(PolicyArn=_policy_arn)["Policy"]["DefaultVersionId"]
_doc = iam.get_policy_version(PolicyArn=_policy_arn, VersionId=_pv)["PolicyVersion"]["Document"]

# Add a statement granting the missing actions (idempotent - checks first)
_missing_actions = [
    "elasticloadbalancing:DescribeListenerAttributes",
    "elasticloadbalancing:ModifyListenerAttributes",
    "elasticloadbalancing:DescribeTrustStores",
    "elasticloadbalancing:DescribeCapacityReservation",
    "elasticloadbalancing:ModifyCapacityReservation",
    "elasticloadbalancing:ModifyIpPools",
]

# Check if the 'ALBControllerV272Patch' SID already exists to prevent duplication
_sid_exists = False
for statement in _doc.get("Statement", []):
    if statement.get("Sid") == "ALBControllerV272Patch":
        _sid_exists = True
        print("IAM policy statement 'ALBControllerV272Patch' already exists. Skipping addition.")
        break

if not _sid_exists:
    _doc["Statement"].append({
        "Sid": "ALBControllerV272Patch",
        "Effect": "Allow",
        "Action": _missing_actions,
        "Resource": "*",
    })
    print(f"Added 'ALBControllerV272Patch' statement to IAM policy with {len(_missing_actions)} actions.")

# Create a new policy version and set as default
# (IAM policies allow max 5 versions; delete oldest non-default if needed)
_versions = iam.list_policy_versions(PolicyArn=_policy_arn)["Versions"]
if len(_versions) >= 5:
    # Find the oldest non-default version to delete
    _oldest_non_default = None
    _min_create_date = None
    for v in _versions:
        if not v["IsDefaultVersion"]:
            if _min_create_date is None or v["CreateDate"] < _min_create_date:
                _min_create_date = v["CreateDate"]
                _oldest_non_default = v

    if _oldest_non_default:
        iam.delete_policy_version(PolicyArn=_policy_arn, VersionId=_oldest_non_default["VersionId"])
        print(f"Deleted old policy version {_oldest_non_default['VersionId']} to make room for new version.")
    else:
        print("Could not find an old non-default policy version to delete. IAM policy may have maxed out versions.")


# Only create a new version if the document was actually modified or to update the default
# We check if the current document is different from the document we intend to apply
_current_default_doc_statements = iam.get_policy_version(PolicyArn=_policy_arn, VersionId=_pv)["PolicyVersion"]["Document"]['Statement']
_new_doc_statements = _doc.get('Statement', [])

# A simple comparison of statements, can be made more robust if needed
if _new_doc_statements != _current_default_doc_statements:
    iam.create_policy_version(
        PolicyArn=_policy_arn,
        PolicyDocument=json.dumps(_doc),
        SetAsDefault=True,
    )
    print(f"Updated {_policy_name} with a new policy version. New statement count: {len(_doc.get('Statement',[]))}")
else:
    print(f"No changes detected in {_policy_name} policy document. Skipping new version creation.")

# Ensure OIDC provider is associated with the cluster
print("\\nAssociating IAM OIDC provider with the cluster...")
_associate_oidc_cmd = (
    f"eksctl utils associate-iam-oidc-provider "
    f"--cluster={cluster_name} "
    f"--region={region} "
    f"--approve"
)
_associate_oidc_result = subprocess.run(_associate_oidc_cmd, shell=True, capture_output=True, text=True, timeout=120)

if _associate_oidc_result.returncode != 0:
    print(f"eksctl associate-iam-oidc-provider failed with exit code {_associate_oidc_result.returncode}.")
    print("--- stdout ---")
    print(_associate_oidc_result.stdout)
    print("--- stderr ---")
    print(_associate_oidc_result.stderr)
    # Do not raise an error here, as it might already be associated, and eksctl might still return non-zero for existing.
    # The subsequent iamserviceaccount creation will confirm if it truly failed.
else:
    print(_associate_oidc_result.stdout)
    print(_associate_oidc_result.stderr) # eksctl often prints to stderr even on success
    print("IAM OIDC provider association command completed.")

# Re-create the IRSA ServiceAccount to ensure it exists
print("\\nEnsuring IRSA service account 'aws-load-balancer-controller' exists...")

# Capture output from eksctl create iamserviceaccount for better debugging
_create_sa_cmd = (
    f"eksctl create iamserviceaccount "
    f"--cluster={cluster_name} "
    f"--region={region} "
    f"--namespace=kube-system "
    f"--name=aws-load-balancer-controller "
    f"--attach-policy-arn={_policy_arn} "
    f"--approve "
    f"--override-existing-serviceaccounts"
)
_create_sa_result = subprocess.run(_create_sa_cmd, shell=True, capture_output=True, text=True, timeout=600)

if _create_sa_result.returncode != 0:
    print(f"eksctl create iamserviceaccount failed with exit code {_create_sa_result.returncode}.")
    print("--- stdout ---")
    print(_create_sa_result.stdout)
    print("--- stderr ---")
    print(_create_sa_result.stderr)
    raise RuntimeError("Failed to create IRSA service account.")
else:
    print(_create_sa_result.stdout)
    print(_create_sa_result.stderr) # eksctl often prints to stderr even on success
    print("IRSA service account creation command completed.")

print("Verifying ServiceAccount creation:")
print(sh("kubectl get sa -n kube-system aws-load-balancer-controller", check=True))

# Force controller to pick up the new permissions (IRSA reads on token refresh, ~15 min otherwise)
print("\\nRestarting aws-load-balancer-controller deployment to pick up new IAM permissions...")
sh("kubectl -n kube-system rollout restart deployment aws-load-balancer-controller")
sh("kubectl -n kube-system rollout status deployment aws-load-balancer-controller --timeout=180s")

# Force reconcile
# This step is commented out because the 'inference-public' service is created in Block 20, later in the notebook.
# This command should be run *after* Block 20 to force reconciliation if needed.
# sh(f"kubectl -n {namespace} annotate svc inference-public reconcile-trigger=$(date +%s) --overwrite")
print("\\nController restarted with new permissions. Wait ~2 min then re-run Block 20.")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... IAM policy statement 'ALBControllerV272Patch' already exists. Skipping addition.
No changes detected in AWSLoadBalancerControllerIAMPolicy policy document. Skipping new version creation.

Associating IAM OIDC provider with the cluster...
2026-09-30 07:44:12 [ℹ]  IAM Open ID Connect provider is already associated with cluster "eks-lab-demo" in "us-east-2"


IAM OIDC provider association command completed.

Ensuring IRSA service account 'aws-load-balancer-controller' exists...
2026-09-30 07:44:12 [ℹ]  1 existing iamserviceaccount(s) (kube-system/aws-load-balancer-controller) will be excluded
2026-09-30 07:44:12 [ℹ]  1 iamserviceaccount (kube-system/aws-load-balancer-controller) was excluded (based on the include/exclude rules)
2026-09-30 07:44:12 [!]  metadata of serviceaccounts that exist in Kubernetes will be updated, as --override-existing-serviceaccounts was set
2026-09-30 07:44:12 [ℹ]  no tasks


IRSA service accoun

To diagnose why the `aws-load-balancer-controller` deployment is not becoming ready, let's inspect the status, events, and logs of the controller's pods. This will help us identify any startup failures, permission errors, or other issues.

In [65]:
%%jshell

runPy("""
print('--- Diagnosing aws-load-balancer-controller status ---')

# Get the status of the controller's pods
print('\\n=== kubectl get pods -n kube-system -l app.kubernetes.io/name=aws-load-balancer-controller ===')
print(sh('kubectl get pods -n kube-system -l app.kubernetes.io/name=aws-load-balancer-controller', check=False))

# Describe the controller's pods for detailed events and conditions
print('\\n=== kubectl describe pods -n kube-system -l app.kubernetes.io/name=aws-load-balancer-controller ===')
print(sh('kubectl describe pods -n kube-system -l app.kubernetes.io/name=aws-load-balancer-controller', check=False))

# Fetch logs from all containers in the controller's pods
print('\\n=== kubectl logs -n kube-system -l app.kubernetes.io/name=aws-load-balancer-controller --tail=50 --all-containers ===')
_logs_result = subprocess.run(
    'kubectl -n kube-system logs -l app.kubernetes.io/name=aws-load-balancer-controller --tail=50 --all-containers',
    shell=True, capture_output=True, text=True
)
if _logs_result.stdout:
    print('--- STDOUT ---')
    print(_logs_result.stdout)
if _logs_result.stderr:
    print('--- STDERR ---')
    print(_logs_result.stderr)
if not _logs_result.stdout and not _logs_result.stderr:
    print("No logs found for 'aws-load-balancer-controller' pods in kube-system namespace.")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... --- Diagnosing aws-load-balancer-controller status ---

=== kubectl get pods -n kube-system -l app.kubernetes.io/name=aws-load-balancer-controller ===
NAME                                            READY   STATUS    RESTARTS   AGE
aws-load-balancer-controller-7fdf97b86d-bzrm9   1/1     Running   0          4m10s
aws-load-balancer-controller-7fdf97b86d-pbd6l   1/1     Running   0          3m59s

=== kubectl describe pods -n kube-system -l app.kubernetes.io/name=aws-load-balancer-controller ===
Name:                 aws-load-balancer-controller-7fdf97b86d-bzrm9
Namespace:            kube-system
Priority:             2000000000
Priority Class Name:  system-cluster-critical
Service Account:      aws-load-balancer-controller
Node:                 ip-192-168-40-177.us-east-2.compute.internal/192.168.40.177
Start Time:           Wed, 30 Sep 2026 07:44:14 +0000
Labels:               app.kubernetes.io/instance=aws-load-balance

The previous diagnostics showed no `aws-load-balancer-controller` pods. Let's get more details directly from the deployment and its ReplicaSets to understand why pods are not being created or are failing to start.

In [66]:
%%jshell

runPy("""
print('--- Deeper Dive: aws-load-balancer-controller Deployment Status ---')

# Get detailed status of the deployment
print('\\n=== kubectl get deployment -n kube-system aws-load-balancer-controller -o yaml ===')
print(sh('kubectl get deployment -n kube-system aws-load-balancer-controller -o yaml', check=False))

# Describe the deployment for events
print('\\n=== kubectl describe deployment -n kube-system aws-load-balancer-controller ===')
print(sh('kubectl describe deployment -n kube-system aws-load-balancer-controller', check=False))

# Get ReplicaSet status for the controller
print('\\n=== kubectl get replicaset -n kube-system -l app.kubernetes.io/name=aws-load-balancer-controller ===')
print(sh('kubectl get replicaset -n kube-system -l app.kubernetes.io/name=aws-load-balancer-controller', check=False))

# Describe the ReplicaSet for more detailed events
print('\\n=== kubectl describe replicaset -n kube-system -l app.kubernetes.io/name=aws-load-balancer-controller ===')
print(sh('kubectl describe replicaset -n kube-system -l app.kubernetes.io/name=aws-load-balancer-controller', check=False))
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... --- Deeper Dive: aws-load-balancer-controller Deployment Status ---

=== kubectl get deployment -n kube-system aws-load-balancer-controller -o yaml ===
apiVersion: apps/v1
kind: Deployment
metadata:
  annotations:
    deployment.kubernetes.io/revision: "2"
    meta.helm.sh/release-name: aws-load-balancer-controller
    meta.helm.sh/release-namespace: kube-system
  creationTimestamp: "2026-09-30T07:41:15Z"
  generation: 2
  labels:
    app.kubernetes.io/instance: aws-load-balancer-controller
    app.kubernetes.io/managed-by: Helm
    app.kubernetes.io/name: aws-load-balancer-controller
    app.kubernetes.io/version: v3.5.0
    helm.sh/chart: aws-load-balancer-controller-3.5.0
  name: aws-load-balancer-controller
  namespace: kube-system
  resourceVersion: "14111"
  uid: 39f508bd-300d-4e6d-8d79-a4bc1105b92f
spec:
  progressDeadlineSeconds: 600
  replicas: 2
  revisionHistoryLimit: 10
  selector:
    matchLabels:
      ap

In [68]:
%%jshell

runPy("""
# Block 20 - LoadBalancer Service pointing at the green pods
# Requires Block 19b (AWS Load Balancer Controller) to have run successfully,
# otherwise the Service will sit at <pending> forever.
#
# If you previously ran this cell before installing the controller, delete the
# stale Service first so the controller sees a fresh CREATE event to reconcile
# (a stale "unchanged" apply does NOT trigger reconciliation).
print("Deleting any prior inference-public Service so the controller reconciles fresh...")
print(sh(f"kubectl -n {namespace} delete svc inference-public --ignore-not-found"))
time.sleep(3)

lb_yaml = {
    "apiVersion": "v1", "kind": "Service",
    "metadata": {
        "name": "inference-public", "namespace": namespace,
        "annotations": {
            "service.beta.kubernetes.io/aws-load-balancer-type":            "external",
            "service.beta.kubernetes.io/aws-load-balancer-nlb-target-type": "ip",
            "service.beta.kubernetes.io/aws-load-balancer-scheme":          "internet-facing",
        },
    },
    "spec": {
        "type": "LoadBalancer",
        "selector": {"app": "inference", "color": "green"},
        "ports": [{"port": 80, "targetPort": APP_PORT, "protocol": "TCP"}],
    },
}
p = work_dir / "lb.yaml"
with open(p, "w") as f: yaml.safe_dump(lb_yaml, f)
print(sh(f"kubectl apply -f {p}"))

# Poll for the NLB hostname - up to 5 min (NLB provisioning is slower than ALB)
print("\\nWaiting for LoadBalancer hostname (up to 5 min)...")
hostname = ""
for i in range(60):
    hostname = sh(
        f"kubectl -n {namespace} get svc inference-public "
        f"-o jsonpath='{{.status.loadBalancer.ingress[0].hostname}}'",
        check=False,
    )
    if hostname:
        print(f"NLB hostname: {hostname}")
        break
    time.sleep(5)
    if i % 6 == 5:
        print(f"  ...still waiting ({(i+1)*5}s elapsed)")
else:
    print("LoadBalancer did not provision in 5 minutes.")
    print(f"Diagnose with:")
    print(f"  kubectl -n {namespace} describe svc inference-public")
    print(f"  kubectl -n kube-system logs -l app.kubernetes.io/name=aws-load-balancer-controller --tail=50")
    print(f"  aws elbv2 describe-load-balancers --region {region}")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Deleting any prior inference-public Service so the controller reconciles fresh...

service/inference-public created

Waiting for LoadBalancer hostname (up to 5 min)...
NLB hostname: k8s-mllab-inferenc-e1fb76a7f2-007345ade7cb8a77.elb.us-east-2.amazonaws.com


In [69]:
%%jshell

runPy("""
# Block 20b - "Open the deployed app" cell
# This is the actual public URL of the inference service running on your cluster.
# Re-run any time to refresh the hostname lookup + rebuild the clickable button.
from IPython.display import display, Markdown, HTML

# Re-resolve the hostname in case Block 20 was run in a different session.
try:
    _host = sh(
        f"kubectl -n {namespace} get svc inference-public "
        f"-o jsonpath='{{.status.loadBalancer.ingress[0].hostname}}'",
        check=False,
    ).strip()
except Exception:
    _host = ""

if _host:
    _app_url = f"http://{_host}"
    _elb_console = (
        f"https://{region}.console.aws.amazon.com/ec2/home?region={region}"
        f"#LoadBalancers:"
    )
    display(Markdown(
        f"### Public inference endpoint\\n"
        f"- **URL:** `{_app_url}`\\n"
        f"- **Selector:** points at the `green` pods (post-cutover)\\n"
    ))
    _btn_green = (
        "display:inline-block;margin-right:10px;padding:12px 22px;background:#0a7d3b;"
        "color:#fff;text-decoration:none;border-radius:4px;font-weight:bold;font-size:15px;"
    )
    _btn_dark = (
        "display:inline-block;padding:12px 22px;background:#232F3E;"
        "color:#fff;text-decoration:none;border-radius:4px;font-weight:bold;"
    )
    _html = (
        '<div style="padding:10px 0;">'
        f'<a href="{_app_url}" target="_blank" style="{_btn_green}">'
        '▶ Open deployed app in browser</a>'
        f'<a href="{_elb_console}" target="_blank" style="{_btn_dark}">'
        '▶ NLB in EC2 Console</a>'
        '<div style="margin-top:8px;color:#666;font-size:12px;">'
        'First click may 5xx for 30-60s while NLB DNS propagates. '
        'Retry until you see the app text.'
        '</div></div>'
    )
    display(HTML(_html))
else:
    display(Markdown(
        "⏳ **LoadBalancer hostname not available yet.** "
        "Re-run Block 20 first, wait ~2 min, then re-run this cell."
    ))
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... <IPython.core.display.Markdown object>
<IPython.core.display.HTML object>


### Diagnosing LoadBalancer Provisioning Failure

It appears the LoadBalancer did not provision successfully in Block 20. Let's run the suggested diagnostic commands to investigate.

In [71]:
%%jshell

runPy("""
print('--- kubectl get svc inference-public -o yaml ---')
print(sh(f'kubectl -n {namespace} get svc inference-public -o yaml', check=False))

print('\\n--- kubectl describe svc inference-public ---')
print(sh(f'kubectl -n {namespace} describe svc inference-public', check=False))

print('\\n--- kubectl logs aws-load-balancer-controller (last 50 lines from all containers) ---')
# For logs, we want to see both stdout and stderr directly, as sh() might suppress empty stdout
_logs_result = subprocess.run(
    f'kubectl -n kube-system logs -l app.kubernetes.io/name=aws-load-balancer-controller --tail=50 --all-containers',
    shell=True, capture_output=True, text=True
)
print(_logs_result.stdout)
print(_logs_result.stderr)

print('\\n--- aws elbv2 describe-load-balancers ---')
print(sh(f'aws elbv2 describe-load-balancers --region {region}'))
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... --- kubectl get svc inference-public -o yaml ---
apiVersion: v1
kind: Service
metadata:
  annotations:
    kubectl.kubernetes.io/last-applied-configuration: |
      {"apiVersion":"v1","kind":"Service","metadata":{"annotations":{"service.beta.kubernetes.io/aws-load-balancer-nlb-target-type":"ip","service.beta.kubernetes.io/aws-load-balancer-scheme":"internet-facing","service.beta.kubernetes.io/aws-load-balancer-type":"external"},"name":"inference-public","namespace":"ml-lab"},"spec":{"ports":[{"port":80,"protocol":"TCP","targetPort":5678}],"selector":{"app":"inference","color":"green"},"type":"LoadBalancer"}}
    service.beta.kubernetes.io/aws-load-balancer-nlb-target-type: ip
    service.beta.kubernetes.io/aws-load-balancer-scheme: internet-facing
    service.beta.kubernetes.io/aws-load-balancer-type: external
  creationTimestamp: "2026-09-30T07:49:19Z"
  finalizers:
  - service.k8s.aws/resources
  name: inference-publ

In [72]:
%%jshell

runPy("""
print('--- Running enhanced diagnostic commands ---')

print('\\n--- kubectl get svc inference-public -o yaml ---')
svc_get_output = sh(f'kubectl -n {namespace} get svc inference-public -o yaml', check=False)
if svc_get_output:
    print(svc_get_output)
else:
    print("Service 'inference-public' not found or no output from 'kubectl get svc'.")

print('\\n--- kubectl describe svc inference-public ---')
svc_describe_output = sh(f'kubectl -n {namespace} describe svc inference-public', check=False)
if svc_describe_output:
    print(svc_describe_output)
else:
    print("Service 'inference-public' not found or no output from 'kubectl describe svc'.")

print('\\n--- kubectl get deployment -n kube-system aws-load-balancer-controller ---')
deployment_output = sh('kubectl -n kube-system get deployment aws-load-balancer-controller', check=False)
if deployment_output:
    print(deployment_output)
else:
    print("Deployment 'aws-load-balancer-controller' not found or no output.")

print('\\n--- kubectl get pods -n kube-system -l app.kubernetes.io/name=aws-load-balancer-controller ---')
pod_list_output = sh('kubectl -n kube-system get pods -l app.kubernetes.io/name=aws-load-balancer-controller', check=False)
if pod_list_output:
    print(pod_list_output)
else:
    print("No 'aws-load-balancer-controller' pods found or no output.")

print('\\n--- kubectl describe pods -n kube-system -l app.kubernetes.io/name=aws-load-balancer-controller ---')
pod_describe_output = sh('kubectl -n kube-system describe pods -l app.kubernetes.io/name=aws-load-balancer-controller', check=False)
if pod_describe_output:
    print(pod_describe_output)
else:
    print("No detailed description for 'aws-load-balancer-controller' pods or no output.")


print('\\n--- kubectl logs aws-load-balancer-controller (last 50 lines from all containers) ---')
_logs_result = subprocess.run(
    f'kubectl -n kube-system logs -l app.kubernetes.io/name=aws-load-balancer-controller --tail=50 --all-containers',
    shell=True, capture_output=True, text=True
)
if _logs_result.stdout:
    print("--- STDOUT ---")
    print(_logs_result.stdout)
if _logs_result.stderr:
    print("--- STDERR ---")
    print(_logs_result.stderr)
if not _logs_result.stdout and not _logs_result.stderr:
    print("No logs found for 'aws-load-balancer-controller' pods in kube-system namespace.")


print('\\n--- aws elbv2 describe-load-balancers ---')
elbv2_output = sh(f'aws elbv2 describe-load-balancers --region {region}', check=False)
if elbv2_output:
    print(elbv2_output)
else:
    print("No AWS ELBv2 Load Balancers found or no output.")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... --- Running enhanced diagnostic commands ---

--- kubectl get svc inference-public -o yaml ---
apiVersion: v1
kind: Service
metadata:
  annotations:
    kubectl.kubernetes.io/last-applied-configuration: |
      {"apiVersion":"v1","kind":"Service","metadata":{"annotations":{"service.beta.kubernetes.io/aws-load-balancer-nlb-target-type":"ip","service.beta.kubernetes.io/aws-load-balancer-scheme":"internet-facing","service.beta.kubernetes.io/aws-load-balancer-type":"external"},"name":"inference-public","namespace":"ml-lab"},"spec":{"ports":[{"port":80,"protocol":"TCP","targetPort":5678}],"selector":{"app":"inference","color":"green"},"type":"LoadBalancer"}}
    service.beta.kubernetes.io/aws-load-balancer-nlb-target-type: ip
    service.beta.kubernetes.io/aws-load-balancer-scheme: internet-facing
    service.beta.kubernetes.io/aws-load-balancer-type: external
  creationTimestamp: "2026-09-30T07:49:19Z"
  finalizers:
  - se

In [73]:
%%jshell

runPy("""
# Block 21 - Call the public endpoint
# DNS propagation for the NLB name takes an additional 30-60s. We retry.
if hostname:
    print(f"Calling http://{hostname} ...")
    for attempt in range(6):
        r = subprocess.run(
            f"curl -s --max-time 10 http://{hostname}",
            shell=True, capture_output=True, text=True,
        )
        if r.returncode == 0 and r.stdout.strip():
            print(f"Response: {r.stdout.strip()}")
            break
        print(f"  attempt {attempt+1}: {'timeout' if r.returncode else 'no body'} - retrying in 15s")
        time.sleep(15)
    else:
        print("Endpoint not responding yet. Wait 60s and try `curl http://{hostname}` from your terminal.")
else:
    print("No hostname available - skipping curl test.")

print("\\n>>> Re-run Block 20b for a clickable browser button to the same URL.")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Calling http://k8s-mllab-inferenc-e1fb76a7f2-007345ade7cb8a77.elb.us-east-2.amazonaws.com ...
  attempt 1: timeout - retrying in 15s
Response: inference-server-v3-green

>>> Re-run Block 20b for a clickable browser button to the same URL.


## 8. Cleanup — guarded

**Set `CLEANUP = True` in Block 3, then re-run just the setup + this section.** The teardown is ordered:

1. Delete K8s Services first — releases the AWS NLB before the cluster is gone (otherwise the NLB becomes an orphaned resource you have to clean up manually)
2. Delete all K8s workloads in `ml-lab` + the namespace itself
3. Wait 30s for the NLB deletion to complete
4. `eksctl delete cluster` — removes control plane, node group, VPC, IAM roles, CloudFormation stacks

Total teardown time: ~10 min.

In [75]:
%%jshell

runPy("""
# Block 22 - Guarded teardown
if not CLEANUP:
    print("CLEANUP is False. Skipping teardown.")
    print("To clean up: set CLEANUP = True in Block 3 and re-run this cell.")
else:
    # --- 1. Delete Services first (releases the NLB) --------------------------
    print("Step 1/5: deleting Kubernetes Services (releases the NLB)...")
    sh(f"kubectl -n {namespace} delete svc inference-public --ignore-not-found --wait=false",
       check=False)
    sh(f"kubectl -n {namespace} delete svc inference-svc    --ignore-not-found --wait=false",
       check=False)
    print("Services deletion initiated.")

    # --- 2. Delete workloads + namespace --------------------------------------
    # --timeout caps how long we wait for finalizers. If a Deployment gets stuck,
    # we prefer to fail fast rather than hang the notebook indefinitely.
    print("\\nStep 2/5: deleting workloads and namespace...")
    sh(f"kubectl -n {namespace} delete hpa inference-hpa               --ignore-not-found",
       check=False)
    sh(f"kubectl -n {namespace} delete deployment --all                --ignore-not-found",
       check=False)
    sh(f"kubectl delete namespace {namespace} --ignore-not-found --timeout=180s",
       check=False)
    print("Workloads deleted.")

    # --- 3. Wait for the NLB to actually be gone (not just deletion request) --
    # If we proceed to eksctl delete while the NLB is still tearing down, the VPC
    # deletion fails because ENIs are still attached. We poll for up to 3 min.
    print("\\nStep 3/5: waiting for NLB to fully release (ENIs detach)...")
    for i in range(18):
        try:
            lbs = elbv2.describe_load_balancers()["LoadBalancers"]
            # NLBs created by K8s Service have name like "k8s-<ns>-<svc>-<hash>"
            k8s_lbs = [lb for lb in lbs if lb["LoadBalancerName"].startswith("k8s-")]
            if not k8s_lbs:
                print(f"  All k8s-* NLBs released (elapsed {i*10}s).")
                break
            print(f"  {len(k8s_lbs)} k8s-* NLB(s) still present, waiting... (elapsed {i*10}s)")
        except Exception as e:
            print(f"  Warning: could not list load balancers: {e}")
            break
        time.sleep(10)
    else:
        print("  Warning: NLB(s) still present after 3 min. Continuing anyway.")
        print("  If eksctl delete fails complaining about ENIs, delete NLBs manually first.")

    # --- 4. eksctl delete cluster ---------------------------------------------
    print(f"\\nStep 4/5: deleting cluster '{cluster_name}' - this takes ~10 min...")
    print("-" * 60)
    rc = sh_stream(
        f"eksctl delete cluster --name {cluster_name} --region {region} --wait --disable-nodegroup-eviction",
        timeout=1200,
    )
    if rc == 0:
        print(f"\\nCluster '{cluster_name}' fully deleted.")
    else:
        print(f"\\neksctl delete returned {rc}. Cluster deletion may still be in progress.")
        print(f"Manual check:")
        print(f"  aws cloudformation list-stacks --region {region} ")
        print(f"    --stack-status-filter DELETE_FAILED ROLLBACK_COMPLETE")
        print("Any stack in DELETE_FAILED state needs manual delete via AWS console.")

    # --- 5. Confirm no CloudFormation stacks left behind ----------------------
    print("\\nStep 5/5: verifying CloudFormation stacks are gone...")
    stacks = cfn.list_stacks(
        StackStatusFilter=["CREATE_COMPLETE","UPDATE_COMPLETE","ROLLBACK_COMPLETE","DELETE_FAILED"]
    )["StackSummaries"]
    lingering = [s for s in stacks if s["StackName"].startswith(f"eksctl-{cluster_name}")]
    if lingering:
        print(f"  {len(lingering)} lingering stack(s):")
        for s in lingering:
            print(f"    - {s['StackName']}  [{s['StackStatus']}]")
    else:
        print("  No lingering eksctl stacks. Fully cleaned up.")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... CLEANUP is False. Skipping teardown.
To clean up: set CLEANUP = True in Block 3 and re-run this cell.


In [76]:
%%jshell

runPy("""
# Block 23 - Verify nothing remains
print("Remaining EKS clusters in region:")
print(sh(f"aws eks list-clusters --region {region}"))
print()
print("Remaining LoadBalancers:")
print(sh(
    f"aws elbv2 describe-load-balancers --region {region} "
    f"--query 'LoadBalancers[?contains(LoadBalancerName, `k8s-`)].[LoadBalancerName,State.Code]' "
    f"--output table",
    check=False,
))
print()
print("Remaining CloudFormation stacks with 'eksctl' prefix:")
print(sh(
    f"aws cloudformation list-stacks --region {region} "
    f"--stack-status-filter CREATE_COMPLETE ROLLBACK_COMPLETE DELETE_FAILED "
    f"--query 'StackSummaries[?starts_with(StackName, `eksctl-`)].[StackName,StackStatus]' "
    f"--output table",
    check=False,
))
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Remaining EKS clusters in region:
{
    "clusters": [
        "eks-lab-demo"
    ]
}

Remaining LoadBalancers:
---------------------------------------------
|           DescribeLoadBalancers           |
+---------------------------------+---------+
|  k8s-mllab-inferenc-e1fb76a7f2  |  active |
+---------------------------------+---------+

Remaining CloudFormation stacks with 'eksctl' prefix:
-------------------------------------------------------------------------------------------------------------
|                                                ListStacks                                                 |
+----------------------------------------------------------------------------------------+------------------+
|  eksctl-eks-lab-demo-addon-iamserviceaccount-kube-system-aws-load-balancer-controller  |  CREATE_COMPLETE |
|  eksctl-eks-lab-demo-nodegroup-ng-primary                                              |  CREA

## 9. Final Checklist

Before you close this notebook:

- [ ] Section 8 ran with `CLEANUP = True` and Block 23 showed empty results
- [ ] AWS Console → EKS: no `eks-lab-*` clusters
- [ ] AWS Console → EC2 → Load Balancers: no `k8s-*` NLBs
- [ ] AWS Console → CloudFormation: no `eksctl-*` stacks (any state)
- [ ] AWS Console → EC2 → Instances: no orphaned worker nodes

Anything still there is a billable resource. Delete manually via the console.

## What this notebook demonstrated

| Section | K8s objects created | AWS resources created | Real service used |
|---------|--------------------|-----------------------|-------------------|
| 2       | —                                                             | EKS cluster, VPC, IAM, 1x t3.medium | EKS + CloudFormation |
| 3       | Namespace, ConfigMap, Secret, Deployment (v1), Service        | —                                   | — |
| 4       | metrics-server, HPA, load-generator pod                       | (autoscaler may add EC2)            | — |
| 5       | Rolling update v1 → v2                                        | —                                   | — |
| 6       | Second Deployment (green/v3), Service selector patch          | —                                   | — |
| 7       | LoadBalancer Service                                          | AWS Network Load Balancer           | ELBv2 |
| 8       | Everything deleted                                            | Everything deleted                  | CloudFormation |

## References

- [Amazon EKS User Guide](https://docs.aws.amazon.com/eks/latest/userguide/)
- [eksctl docs](https://eksctl.io/)
- [Kubernetes concepts](https://kubernetes.io/docs/concepts/)
- [AWS Load Balancer Controller](https://kubernetes-sigs.github.io/aws-load-balancer-controller/) — for Ingress → ALB instead of one NLB per Service

In [77]:
%%jshell

runPy("""
import datetime, pytz;
print("Current Time in IST:", datetime.datetime.now(pytz.utc).astimezone(pytz.timezone('Asia/Kolkata')).strftime('%Y-%m-%d %H:%M:%S'))
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Current Time in IST: 2026-09-30 13:23:33


In [78]:
%%jshell

runPy("""
print('Checking rollout status of aws-load-balancer-controller deployment...')
sh_stream("kubectl -n kube-system rollout status deployment aws-load-balancer-controller --timeout=300s")
print("\\nIf the rollout succeeded, you can now proceed to Block 20 to provision the LoadBalancer Service. If it failed, please let me know, and we can run further diagnostics.")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Checking rollout status of aws-load-balancer-controller deployment...
deployment "aws-load-balancer-controller" successfully rolled out

If the rollout succeeded, you can now proceed to Block 20 to provision the LoadBalancer Service. If it failed, please let me know, and we can run further diagnostics.


In [79]:
%%jshell

runPy("""
import subprocess

def sh(cmd, check=True, capture=True, timeout=None, env=None):
    # A placeholder for ALLOW_MUTATIONS, as it might not be globally available here.
    # For this specific command, we want it to run, so we'll assume mutations are allowed or this is a read-only command.
    # If this causes issues with other commands, a more robust solution involving passing ALLOW_MUTATIONS would be needed.
    r = subprocess.run(cmd, shell=True, capture_output=capture, text=True, timeout=timeout, env=env)
    if check and r.returncode != 0:
        raise RuntimeError(
            f"Command failed (exit {r.returncode}): {cmd}\\n"
            f"--- stdout ---\\n{r.stdout}\\n"
            f"--- stderr ---\\n{r.stderr}"
        )
    return (r.stdout or "").strip()

print('Confirming aws-load-balancer-controller deployment status...')
_status = sh('kubectl get deployment -n kube-system aws-load-balancer-controller -o jsonpath="{.status.conditions[?(@.type==\"Available\")].status}"', check=False)

if _status == 'True':
    print('\\n✅ The aws-load-balancer-controller deployment is Available. You can now proceed to Block 20 to provision the LoadBalancer Service.')
else:
    print(f'\\n❌ The aws-load-balancer-controller deployment is not yet Available. Current status: {_status}')
    print('You can re-run the previous cell to wait longer for the rollout or run further diagnostics if it remains stuck.')
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Confirming aws-load-balancer-controller deployment status...

❌ The aws-load-balancer-controller deployment is not yet Available. Current status: 
You can re-run the previous cell to wait longer for the rollout or run further diagnostics if it remains stuck.


In [80]:
%%jshell

runPy("""
import subprocess

def sh(cmd, check=True, capture=True, timeout=None, env=None):
    r = subprocess.run(cmd, shell=True, capture_output=capture, text=True, timeout=timeout, env=env)
    if check and r.returncode != 0:
        raise RuntimeError(
            f"Command failed (exit {r.returncode}): {cmd}\\n"
            f"--- stdout ---\\n{r.stdout}\\n"
            f"--- stderr ---\\n{r.stderr}"
        )
    return (r.stdout or "").strip()

print('Confirming aws-load-balancer-controller deployment status again...')
_status = sh('kubectl get deployment -n kube-system aws-load-balancer-controller -o jsonpath="{.status.conditions[?(@.type==\"Available\")].status}"', check=False)

if _status == 'True':
    print('\\n✅ The aws-load-balancer-controller deployment is Available. You can now proceed to Block 20 to provision the LoadBalancer Service.')
else:
    print(f'\\n❌ The aws-load-balancer-controller deployment is not yet Available. Current status: {_status}')
    print('You can re-run the previous cell to wait longer for the rollout or run further diagnostics if it remains stuck.')
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Confirming aws-load-balancer-controller deployment status again...

❌ The aws-load-balancer-controller deployment is not yet Available. Current status: 
You can re-run the previous cell to wait longer for the rollout or run further diagnostics if it remains stuck.


In [82]:
%%jshell

runPy("""
import subprocess

def sh(cmd, check=True, capture=True, timeout=None, env=None):
    r = subprocess.run(cmd, shell=True, capture_output=capture, text=True, timeout=timeout, env=env)
    if check and r.returncode != 0:
        raise RuntimeError(
            f"Command failed (exit {r.returncode}): {cmd}\\n"
            f"--- stdout ---\\n{r.stdout}\\n"
            f"--- stderr ---\\n{r.stderr}"
        )
    return (r.stdout or "").strip()

print('--- Diagnosing aws-load-balancer-controller status ---')

# Get the status of the controller's pods
print('\\n=== kubectl get pods -n kube-system -l app.kubernetes.io/name=aws-load-balancer-controller ===')
print(sh('kubectl get pods -n kube-system -l app.kubernetes.io/name=aws-load-balancer-controller', check=False))

# Describe the controller's pods for detailed events and conditions
print('\\n=== kubectl describe pods -n kube-system -l app.kubernetes.io/name=aws-load-balancer-controller ===')
print(sh('kubectl describe pods -n kube-system -l app.kubernetes.io/name=aws-load-balancer-controller', check=False))

# Fetch logs from all containers in the controller's pods
print('\\n=== kubectl logs -n kube-system -l app.kubernetes.io/name=aws-load-balancer-controller --tail=50 --all-containers ===')
_logs_result = subprocess.run(
    'kubectl -n kube-system logs -l app.kubernetes.io/name=aws-load-balancer-controller --tail=50 --all-containers',
    shell=True, capture_output=True, text=True
)
if _logs_result.stdout:
    print('--- STDOUT ---')
    print(_logs_result.stdout)
if _logs_result.stderr:
    print('--- STDERR ---')
    print(_logs_result.stderr)
if not _logs_result.stdout and not _logs_result.stderr:
    print("No logs found for 'aws-load-balancer-controller' pods in kube-system namespace.")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... --- Diagnosing aws-load-balancer-controller status ---

=== kubectl get pods -n kube-system -l app.kubernetes.io/name=aws-load-balancer-controller ===
NAME                                            READY   STATUS    RESTARTS   AGE
aws-load-balancer-controller-7fdf97b86d-bzrm9   1/1     Running   0          11m
aws-load-balancer-controller-7fdf97b86d-pbd6l   1/1     Running   0          11m

=== kubectl describe pods -n kube-system -l app.kubernetes.io/name=aws-load-balancer-controller ===
Name:                 aws-load-balancer-controller-7fdf97b86d-bzrm9
Namespace:            kube-system
Priority:             2000000000
Priority Class Name:  system-cluster-critical
Service Account:      aws-load-balancer-controller
Node:                 ip-192-168-40-177.us-east-2.compute.internal/192.168.40.177
Start Time:           Wed, 30 Sep 2026 07:44:14 +0000
Labels:               app.kubernetes.io/instance=aws-load-balancer-co

In [83]:
%%jshell

runPy("""
import subprocess

def sh(cmd, check=True, capture=True, timeout=None, env=None):
    r = subprocess.run(cmd, shell=True, capture_output=capture, text=True, timeout=timeout, env=env)
    if check and r.returncode != 0:
        raise RuntimeError(
            f"Command failed (exit {r.returncode}): {cmd}\\n"
            f"--- stdout ---\\n{r.stdout}\\n"
            f"--- stderr ---\\n{r.stderr}"
        )
    return (r.stdout or "").strip()

print('--- Deeper Dive: aws-load-balancer-controller Deployment Status ---')

# Get detailed status of the deployment
print('\\n=== kubectl get deployment -n kube-system aws-load-balancer-controller -o yaml ===')
print(sh('kubectl get deployment -n kube-system aws-load-balancer-controller -o yaml', check=False))

# Describe the deployment for events
print('\\n=== kubectl describe deployment -n kube-system aws-load-balancer-controller ===')
print(sh('kubectl describe deployment -n kube-system aws-load-balancer-controller', check=False))

# Get ReplicaSet status for the controller
print('\\n=== kubectl get replicaset -n kube-system -l app.kubernetes.io/name=aws-load-balancer-controller ===')
print(sh('kubectl get replicaset -n kube-system -l app.kubernetes.io/name=aws-load-balancer-controller', check=False))

# Describe the ReplicaSet for more detailed events
print('\\n=== kubectl describe replicaset -n kube-system -l app.kubernetes.io/name=aws-load-balancer-controller ===')
print(sh('kubectl describe replicaset -n kube-system -l app.kubernetes.io/name=aws-load-balancer-controller', check=False))
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... --- Deeper Dive: aws-load-balancer-controller Deployment Status ---

=== kubectl get deployment -n kube-system aws-load-balancer-controller -o yaml ===
apiVersion: apps/v1
kind: Deployment
metadata:
  annotations:
    deployment.kubernetes.io/revision: "2"
    meta.helm.sh/release-name: aws-load-balancer-controller
    meta.helm.sh/release-namespace: kube-system
  creationTimestamp: "2026-09-30T07:41:15Z"
  generation: 2
  labels:
    app.kubernetes.io/instance: aws-load-balancer-controller
    app.kubernetes.io/managed-by: Helm
    app.kubernetes.io/name: aws-load-balancer-controller
    app.kubernetes.io/version: v3.5.0
    helm.sh/chart: aws-load-balancer-controller-3.5.0
  name: aws-load-balancer-controller
  namespace: kube-system
  resourceVersion: "14111"
  uid: 39f508bd-300d-4e6d-8d79-a4bc1105b92f
spec:
  progressDeadlineSeconds: 600
  replicas: 2
  revisionHistoryLimit: 10
  selector:
    matchLabels:
      ap

In [84]:
%%jshell

runPy("""
import datetime, pytz;
print("Current Time in IST:", datetime.datetime.now(pytz.utc).astimezone(pytz.timezone('Asia/Kolkata')).strftime('%Y-%m-%d %H:%M:%S'))
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Current Time in IST: 2026-09-30 13:25:58
